# Biological BMI Paper — Beta-coefficients of LASSO Models

***by Kengo Watanabe***  

Based on the beta-coefficients of LASSO models for BMI (biological BMIs), variables are visualized and assessed with BMI regression.  

Input:  
* Cleaned metabolomics: 210104_Biological-BMI-paper_RF-imputation_baseline-metDF-with-RF-imputation.tsv  
* MetBMI predictions: 210105_Biological-BMI-paper_baseline-LASSO_metBMI-BothSex.csv  
* MetBMI predictions (female model): 210105_Biological-BMI-paper_baseline-LASSO_metBMI-Female.csv  
* MetBMI predictions (male model): 210105_Biological-BMI-paper_baseline-LASSO_metBMI-Male.csv  
* Beta-coefficients of MetBMI models: 210105_Biological-BMI-paper_baseline-LASSO_metBMI-BothSex_LASSO-bcoefs.tsv  
* Beta-coefficients of MetBMI models (female model): 210105_Biological-BMI-paper_baseline-LASSO_metBMI-Female_LASSO-bcoefs.tsv  
* Beta-coefficients of MetBMI models (male model): 210105_Biological-BMI-paper_baseline-LASSO_metBMI-Male_LASSO-bcoefs.tsv  
* Cleaned clinical labs: 210104_Biological-BMI-paper_RF-imputation_baseline-chemDF-with-RF-imputation.tsv  
* ChemBMI predictions: 210105_Biological-BMI-paper_baseline-LASSO_chemBMI-BothSex.csv  
* ChemBMI predictions (female model): 210105_Biological-BMI-paper_baseline-LASSO_chemBMI-Female.csv  
* ChemBMI predictions (male model): 210105_Biological-BMI-paper_baseline-LASSO_chemBMI-Male.csv  
* Beta-coefficients of ChemBMI models: 210105_Biological-BMI-paper_baseline-LASSO_chemBMI-BothSex_LASSO-bcoefs.tsv  
* Beta-coefficients of ChemBMI models (female model): 210105_Biological-BMI-paper_baseline-LASSO_chemBMI-Female_LASSO-bcoefs.tsv  
* Beta-coefficients of ChemBMI models (male model): 210105_Biological-BMI-paper_baseline-LASSO_chemBMI-Male_LASSO-bcoefs.tsv  
* Cleaned proteomics: 210104_Biological-BMI-paper_RF-imputation_baseline-protDF-with-RF-imputation.tsv  
* ProtBMI predictions: 210105_Biological-BMI-paper_baseline-LASSO_protBMI-BothSex.csv  
* ProtBMI predictions (female model): 210105_Biological-BMI-paper_baseline-LASSO_protBMI-Female.csv  
* ProtBMI predictions (male model): 210105_Biological-BMI-paper_baseline-LASSO_protBMI-Male.csv  
* Beta-coefficients of ProtBMI models: 210105_Biological-BMI-paper_baseline-LASSO_protBMI-BothSex_LASSO-bcoefs.tsv  
* Beta-coefficients of ProtBMI models (female model): 210105_Biological-BMI-paper_baseline-LASSO_protBMI-Female_LASSO-bcoefs.tsv  
* Beta-coefficients of ProtBMI models (male model): 210105_Biological-BMI-paper_baseline-LASSO_protBMI-Male_LASSO-bcoefs.tsv  
* Cleaned combined omics: 210104_Biological-BMI-paper_RF-imputation_baseline-combiDF-with-RF-imputation.tsv  
* CombiBMI predictions: 210105_Biological-BMI-paper_baseline-LASSO_combiBMI-BothSex.csv  
* CombiBMI predictions (female model): 210105_Biological-BMI-paper_baseline-LASSO_combiBMI-Female.csv  
* CombiBMI predictions (male model): 210105_Biological-BMI-paper_baseline-LASSO_combiBMI-Male.csv  
* Beta-coefficients of CombiBMI models: 210105_Biological-BMI-paper_baseline-LASSO_combiBMI-BothSex_LASSO-bcoefs.tsv  
* Beta-coefficients of CombiBMI models (female model): 210105_Biological-BMI-paper_baseline-LASSO_combiBMI-Female_LASSO-bcoefs.tsv  
* Beta-coefficients of CombiBMI models (male model): 210105_Biological-BMI-paper_baseline-LASSO_combiBMI-Male_LASSO-bcoefs.tsv   

Output:  
* Figure 2  
* Supplementary Figure 2  
* Supplementary Figure 4  
* Intermediate tables for other notebook (regression summary)  

Original notebook:  
* 210105_Biological-BMI-paper_LASSO-bcoef.ipynb  
* 210823_Biological-BMI-paper_LASSO-bcoef(figure-layout).ipynb  

In [ ]:
import numpy as np
import pandas as pd
import scipy.stats as stats
import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns
#For Arial font
#!conda install -c conda-forge -y mscorefonts
import warnings
warnings.filterwarnings('ignore')
from IPython.display import display
import time

from sklearn.preprocessing import StandardScaler
#!conda install -c conda-forge matplotlib-venn
from matplotlib_venn import venn3, venn3_circles, venn2, venn2_circles
import statsmodels.formula.api as smf
from statsmodels.stats import multitest as multi
import matplotlib.patches as mpatches

!conda list

In [ ]:
covarL = ['log_BaseBMI', 'Sex', 'BaseAge', 'PC1', 'PC2', 'PC3', 'PC4', 'PC5']
#-> Attention in handling: race is also included as reference in the original dataframe.

## 1. Metabolomics

### 1-1. Prepare variables

> Variables are standardized for OLS.

In [ ]:
#Import the cleaned baseline dataframe for LASSO
fileDir = './ExportData/'
fileName = '210104_Biological-BMI-paper_RF-imputation_baseline-metDF-with-RF-imputation.tsv'
tempDF = pd.read_csv(fileDir+fileName, sep='\t', dtype={'public_client_id': str}).set_index('public_client_id')

#Splitting dataset into sex-dependent cohorts
metDF_F = tempDF.loc[tempDF['Sex']=='F']
metDF_M = tempDF.loc[tempDF['Sex']=='M']
metDF_B = tempDF#Not copy just rename
print('(Female, Male, Both): (', len(metDF_F.index), ', ', len(metDF_M.index), ', ', len(metDF_B.index), ')')

In [ ]:
#Female models

#Import biological BMI
fileDir = './ExportData/'
fileName = '210105_Biological-BMI-paper_baseline-LASSO_metBMI-Female.csv'
tempDF = pd.read_csv(fileDir+fileName, dtype={'public_client_id': str}).set_index('public_client_id')
metDF_F['log_BaseMetBMI'] = tempDF['log_BaseMetBMI']

#Z-score standardization
tempDF = metDF_F.drop(columns=['Sex', 'Race'])
tempL1 = tempDF.index.tolist()
tempL2 = tempDF.columns.tolist()
scaler = StandardScaler(copy=True, with_mean=True, with_std=True)
tempA = scaler.fit_transform(tempDF)#Column direction
tempDF = pd.DataFrame(data=tempA, index=tempL1, columns=tempL2)
tempDF['Sex'] = metDF_F['Sex']

#Update
metDF_F = tempDF
display(metDF_F.describe(include='all'))

In [ ]:
#Male model

#Import biological BMI
fileDir = './ExportData/'
fileName = '210105_Biological-BMI-paper_baseline-LASSO_metBMI-Male.csv'
tempDF = pd.read_csv(fileDir+fileName, dtype={'public_client_id': str}).set_index('public_client_id')
metDF_M['log_BaseMetBMI'] = tempDF['log_BaseMetBMI']

#Z-score standardization
tempDF = metDF_M.drop(columns=['Sex', 'Race'])
tempL1 = tempDF.index.tolist()
tempL2 = tempDF.columns.tolist()
scaler = StandardScaler(copy=True, with_mean=True, with_std=True)
tempA = scaler.fit_transform(tempDF)#Column direction
tempDF = pd.DataFrame(data=tempA, index=tempL1, columns=tempL2)
tempDF['Sex'] = metDF_M['Sex']

#Update
metDF_M = tempDF
display(metDF_M.describe(include='all'))

In [ ]:
#Both sex model

#Import biological BMI
fileDir = './ExportData/'
fileName = '210105_Biological-BMI-paper_baseline-LASSO_metBMI-BothSex.csv'
tempDF = pd.read_csv(fileDir+fileName, dtype={'public_client_id': str}).set_index('public_client_id')
metDF_B['log_BaseMetBMI'] = tempDF['log_BaseMetBMI']

#Z-score standardization
tempDF = metDF_B.drop(columns=['Sex', 'Race'])
tempL1 = tempDF.index.tolist()
tempL2 = tempDF.columns.tolist()
scaler = StandardScaler(copy=True, with_mean=True, with_std=True)
tempA = scaler.fit_transform(tempDF)#Column direction
tempDF = pd.DataFrame(data=tempA, index=tempL1, columns=tempL2)
tempDF['Sex'] = metDF_B['Sex']

#Update
metDF_B = tempDF
display(metDF_B.describe(include='all'))

### 1-2. Correlation b/w all pairwise variables

In [ ]:
#Female and male cohorts
tempL = [item for sublist in [covarL, ['log_BaseMetBMI']] for item in sublist]

print('Female cohort:')
#Compute correlation matrix
tempDF1 = metDF_F.drop(columns=tempL)
tempDF1 = tempDF1.corr(method='pearson')
print('• Combinations:', int(len(tempDF1)*(len(tempDF1)-1)/2))

#Extract lower triangle matrix
tempDF1 = tempDF1.where(np.tril(np.ones(tempDF1.shape), k=-1).astype(np.bool), other=np.nan)
tempDF1.index.set_names('Variable1', inplace=True)
tempDF1 = tempDF1.reset_index().melt(var_name='Variable2', value_name='Pearson_r', id_vars=['Variable1'])
tempDF1 = tempDF1.dropna()
print('• nrows:', len(tempDF1))
print('• |Pearson\'s r| > 0.8:', len(tempDF1.loc[abs(tempDF1['Pearson_r'])>0.8]))
display(tempDF1.loc[abs(tempDF1['Pearson_r'])>0.8])

print('Male cohort:')
#Compute correlation matrix
tempDF2 = metDF_M.drop(columns=tempL)
tempDF2 = tempDF2.corr(method='pearson')
print('• Combinations:', int(len(tempDF2)*(len(tempDF2)-1)/2))

#Extract lower triangle matrix
tempDF2 = tempDF2.where(np.tril(np.ones(tempDF2.shape), k=-1).astype(np.bool), other=np.nan)
tempDF2.index.set_names('Variable1', inplace=True)
tempDF2 = tempDF2.reset_index().melt(var_name='Variable2', value_name='Pearson_r', id_vars=['Variable1'])
tempDF2 = tempDF2.dropna()
print('• nrows:', len(tempDF2))
print('• |Pearson\'s r| > 0.8:', len(tempDF2.loc[abs(tempDF2['Pearson_r'])>0.8]))
display(tempDF2.loc[abs(tempDF2['Pearson_r'])>0.8])

#Distribution
sns.set(style='ticks', font='Arial', context='notebook')
plt.figure(figsize=(4, 3))
sns.distplot(tempDF1['Pearson_r'], color='r', label='Female').set(xlim=(-1, 1), xticks=np.arange(-1, 1.1, 0.5))
sns.distplot(tempDF2['Pearson_r'], color='b', label='Male').set(xlim=(-1, 1), xticks=np.arange(-1, 1.1, 0.5))
sns.despine()
plt.xlabel('Pearson\'s '+r'$r$')
plt.ylabel('Density')
plt.legend(bbox_to_anchor=(1, 0.5), loc='center left', borderaxespad=1)
plt.show()

In [ ]:
#Both sex cohort
tempL = [item for sublist in [covarL, ['log_BaseMetBMI']] for item in sublist]

print('Both sex cohort:')
#Compute correlation matrix
tempDF = metDF_B.drop(columns=tempL)
tempDF = tempDF.corr(method='pearson')
print('• Combinations:', int(len(tempDF)*(len(tempDF)-1)/2))

#Extract lower triangle matrix
tempDF = tempDF.where(np.tril(np.ones(tempDF.shape), k=-1).astype(np.bool), other=np.nan)
tempDF.index.set_names('Variable1', inplace=True)
tempDF = tempDF.reset_index().melt(var_name='Variable2', value_name='Pearson_r', id_vars=['Variable1'])
tempDF = tempDF.dropna()
print('• nrows:', len(tempDF))
print('• |Pearson\'s r| > 0.8:', len(tempDF.loc[abs(tempDF['Pearson_r'])>0.8]))
display(tempDF.loc[abs(tempDF['Pearson_r'])>0.8])

#Distribution
sns.set(style='ticks', font='Arial', context='talk')
plt.figure(figsize=(4, 3))
sns.distplot(tempDF['Pearson_r'], color='b').set(xlim=(-1, 1), xticks=np.arange(-1, 1.1, 0.5))
sns.despine()
plt.ylabel('Density')
plt.xlabel('Pearson\'s '+r'$r$')
plt.axvline(x=tempDF['Pearson_r'].max(), **{'linestyle':'--', 'color':'b'})
plt.axvline(x=tempDF['Pearson_r'].min(), **{'linestyle':'--', 'color':'b'})
plt.show()

### 1-3. Beta-coefficients of LASSO models

In [ ]:
#Female model
#Import the cleaned beta-coefficients of LASSO
fileDir = './ExportData/'
fileName = '210105_Biological-BMI-paper_baseline-LASSO_metBMI-Female_LASSO-bcoefs.tsv'
metBMI_F_bcoefs = pd.read_csv(fileDir+fileName, sep='\t').set_index('Variable')
print('Variables:', len(metBMI_F_bcoefs))

#Variables with non-zero beta-coefficient
tempDF = metBMI_F_bcoefs.loc[metBMI_F_bcoefs['nZeros']!=10]
print('Variables with non-zero beta-coefficient:', len(tempDF),
      '(', len(tempDF)/len(metBMI_F_bcoefs)*100, '%)')

#Variables with non-zero beta-coefficient in all 10 models
tempDF = metBMI_F_bcoefs.loc[metBMI_F_bcoefs['nZeros']==0]
tempDF = tempDF.sort_values(by='MEAN', ascending=False)
print('Variables with non-zero beta-coefficient in all 10 models:', len(tempDF),
      '(', len(tempDF)/len(metBMI_F_bcoefs)*100, '%)')
display(tempDF)

In [ ]:
#Male model
#Import the cleaned beta-coefficients of LASSO
fileDir = './ExportData/'
fileName = '210105_Biological-BMI-paper_baseline-LASSO_metBMI-Male_LASSO-bcoefs.tsv'
metBMI_M_bcoefs = pd.read_csv(fileDir+fileName, sep='\t').set_index('Variable')
print('Variables:', len(metBMI_M_bcoefs))

#Variables with non-zero beta-coefficient
tempDF = metBMI_M_bcoefs.loc[metBMI_M_bcoefs['nZeros']!=10]
print('Variables with non-zero beta-coefficient:', len(tempDF),
      '(', len(tempDF)/len(metBMI_M_bcoefs)*100, '%)')

#Variables with non-zero beta-coefficient in all 10 models
tempDF = metBMI_M_bcoefs.loc[metBMI_M_bcoefs['nZeros']==0]
tempDF = tempDF.sort_values(by='MEAN', ascending=False)
print('Variables with non-zero beta-coefficient in all 10 models:', len(tempDF),
      '(', len(tempDF)/len(metBMI_M_bcoefs)*100, '%)')
display(tempDF)

In [ ]:
#Both sex model
#Import the cleaned beta-coefficients of LASSO
fileDir = './ExportData/'
fileName = '210105_Biological-BMI-paper_baseline-LASSO_metBMI-BothSex_LASSO-bcoefs.tsv'
metBMI_B_bcoefs = pd.read_csv(fileDir+fileName, sep='\t').set_index('Variable')
print('Variables:', len(metBMI_B_bcoefs))

#Variables with non-zero beta-coefficient
tempDF = metBMI_B_bcoefs.loc[metBMI_B_bcoefs['nZeros']!=10]
print('Variables with non-zero beta-coefficient:', len(tempDF),
      '(', len(tempDF)/len(metBMI_B_bcoefs)*100, '%)')

#Variables with non-zero beta-coefficient in all 10 models
tempDF = metBMI_B_bcoefs.loc[metBMI_B_bcoefs['nZeros']==0]
tempDF = tempDF.sort_values(by='MEAN', ascending=False)
print('Variables with non-zero beta-coefficient in all 10 models:', len(tempDF),
      '(', len(tempDF)/len(metBMI_B_bcoefs)*100, '%)')
display(tempDF)

### 1-4. Variables with non-zero beta-coefficient in all 10 models

In [ ]:
#Variables with non-zero beta-coefficient in all 10 models
tempS1 = set(metBMI_F_bcoefs.loc[metBMI_F_bcoefs['nZeros']==0].index)
tempS2 = set(metBMI_M_bcoefs.loc[metBMI_M_bcoefs['nZeros']==0].index)
tempS3 = set(metBMI_B_bcoefs.loc[metBMI_B_bcoefs['nZeros']==0].index)

#Common the robust variables
var_111 = tempS1 & tempS2 & tempS3
print('Common variables with non-zero beta-coefficient in all 10 models:\n', var_111)
var_110 = (tempS1 & tempS2) - var_111
var_101 = (tempS1 & tempS3) - var_111
var_011 = (tempS2 & tempS3) - var_111
var_100 = tempS1 - (var_110 | var_101 | var_111)
var_010 = tempS2 - (var_110 | var_011 | var_111)
var_001 = tempS3 - (var_101 | var_011 | var_111)

#The number of each subset
var_subset = [len(var_100), len(var_010), len(var_110), len(var_001),
              len(var_101), len(var_011), len(var_111)]
print('Each subset:', var_subset)

#Venn diagram
sns.set(font='Arial', context='talk')
venn3(subsets=var_subset, set_labels=('Female', 'Male', 'Both sex'), set_colors=('r', 'b', 'g'), alpha=0.4)
venn3_circles(subsets=var_subset)
plt.title('Robust variables\n—Metabolomics—', fontdict={'fontsize':24})
plt.show()

In [ ]:
#Variables with non-zero beta-coefficient in all 10 models
tempS1 = set(metBMI_F_bcoefs.loc[metBMI_F_bcoefs['nZeros']==0].index)
tempS2 = set(metBMI_M_bcoefs.loc[metBMI_M_bcoefs['nZeros']==0].index)
tempS3 = set(metBMI_B_bcoefs.loc[metBMI_B_bcoefs['nZeros']==0].index)

#Union of the robust variables
tempL = list(tempS1 | tempS2 | tempS3)
print('Union of the variables with non-zero beta-coefficient in all 10 models:', len(tempL))

#Merge (use original bcoefs to rescue ones with zeros in any 10 models)
tempDF1 = metBMI_F_bcoefs.loc[metBMI_F_bcoefs.index.isin(tempL)].drop(columns=['MEAN', 'SD', 'nZeros'])
tempDF2 = metBMI_M_bcoefs.loc[metBMI_M_bcoefs.index.isin(tempL)].drop(columns=['MEAN', 'SD', 'nZeros'])
tempDF3 = metBMI_B_bcoefs.loc[metBMI_B_bcoefs.index.isin(tempL)].drop(columns=['MEAN', 'SD', 'nZeros'])
tempDF = pd.concat([tempDF1, tempDF2, tempDF3], axis=0)
tempDF['Model'] = np.repeat(['Female', 'Male', 'Both sex'], [len(tempDF1), len(tempDF2), len(tempDF3)])

#tidyr::gather in R
tempDF1 = tempDF.reset_index().melt(var_name='LASSO', value_name='bcoef', id_vars=['Variable', 'Model'])

#Order for plot
tempDF2 = tempDF1.loc[tempDF1['Model']=='Both sex'].groupby('Variable', as_index=True).agg({'bcoef': np.mean})
tempDF2 = tempDF2.sort_values(by='bcoef', ascending=False)

#Plot
sns.set(style='ticks', font='Arial', context='talk')
sns.catplot(data=tempDF1, y='Variable', order=tempDF2.index.tolist(), x='bcoef', kind='box',
                hue='Model', palette='Set1', dodge=True, height=35, aspect=0.35,
                showfliers=True, flierprops={'marker':'o', 'markerfacecolor':'gray', 'alpha':0.4})
plt.xlabel(r'$\beta$'+'-coefficient in LASSO models')
plt.ylabel('')
tempA = np.arange(-0.02, 0.041, 0.02)
for coord in tempA:
    plt.axvline(x=coord, **{'linestyle':'--', 'color':'gray'})
for row_i in range(len(tempDF2)):
    if row_i%2 == 0:
        plt.axhspan(ymin=row_i-0.5, ymax=row_i+0.5, facecolor='b', alpha=0.1)
plt.show()

In [ ]:
#Both sex model

#Variables with non-zero beta-coefficient in all 10 models
tempDF = metBMI_B_bcoefs.loc[metBMI_B_bcoefs['nZeros']==0].sort_values(by='MEAN', ascending=False)
tempDF = tempDF.drop(columns=['MEAN', 'SD', 'nZeros'])
print('Variables with non-zero beta-coefficient in all 10 models:', len(tempDF))

#tidyr::gather in R
tempDF1 = tempDF.reset_index().melt(var_name='LASSO', value_name='bcoef', id_vars=['Variable'])

#Plot
sns.set(style='ticks', font='Arial', context='talk')
plt.figure(figsize=(5, 20))
p = sns.boxplot(data=tempDF1, y='Variable', x='bcoef', color='b', dodge=False,
                showfliers=True, flierprops={'marker':'o', 'markerfacecolor':'gray', 'alpha':0.4},
                showcaps=True, notch=False)
p.set(xlim=(-0.046, 0.056), xticks=np.arange(-0.04, 0.06, 0.02))
p.grid(axis='x', linestyle='--', color='black')
sns.despine()
plt.ylabel('')
plt.xlabel(r'$\beta$'+'-coefficient in LASSO model')
for row_i in range(len(tempDF)):
    if row_i%2 == 0:
        plt.axhspan(ymin=row_i-0.5, ymax=row_i+0.5, facecolor='b', alpha=0.2, zorder=0)
##Save
fileDir = './ExportFigures/'
fileName = '210823_Biological-BMI-paper_LASSO-bcoef_metBMI-BothSex-bcoef_non-zero-in-all.tif'
plt.gcf().savefig(fileDir+fileName, dpi=300, bbox_inches='tight', pad_inches=0,
                  pil_kwargs={'compression':'tiff_lzw'})
plt.show()

### 1-5. Correlation b/w pairwise variables with non-zero beta coefficient in all 10 models

In [ ]:
#Both sex model
print('Both sex model:')

#Variables with non-zero beta-coefficient in all 10 models
tempL = metBMI_B_bcoefs.loc[metBMI_B_bcoefs['nZeros']==0].index.tolist()

#Compute correlation matrix
tempDF = metDF_B[tempL]
tempDF = tempDF.corr(method='pearson')
print('• Combinations:', int(len(tempDF)*(len(tempDF)-1)/2))

#Extract lower triangle matrix
tempDF = tempDF.where(np.tril(np.ones(tempDF.shape), k=-1).astype(np.bool), other=np.nan)
tempDF.index.set_names('Variable1', inplace=True)
tempDF = tempDF.reset_index().melt(var_name='Variable2', value_name='Pearson_r', id_vars=['Variable1'])
tempDF = tempDF.dropna()
print('• nrows:', len(tempDF))
print('• |Pearson\'s r| > 0.8:', len(tempDF.loc[abs(tempDF['Pearson_r'])>0.8]))
display(tempDF.loc[abs(tempDF['Pearson_r'])>0.8])

#Distribution
sns.set(style='ticks', font='Arial', context='talk')
plt.figure(figsize=(4, 3))
sns.distplot(tempDF['Pearson_r'], color='b').set(xlim=(-1, 1), xticks=np.arange(-1, 1.1, 0.5))
sns.despine()
plt.ylabel('Density')
plt.xlabel('Pearson\'s '+r'$r$')
plt.axvline(x=tempDF['Pearson_r'].max(), **{'linestyle':'--', 'color':'b'})
plt.axvline(x=tempDF['Pearson_r'].min(), **{'linestyle':'--', 'color':'b'})
plt.show()

print('Variables with non-zero beta-coefficient in all 10 models:', len(tempL))
#Clustermap
tempDF = metDF_B[tempL]
tempDF = tempDF.corr(method='pearson')
sns.set(style='ticks', font='Arial', context='notebook')
cmap = sns.diverging_palette(220, 20, as_cmap=True)
cm = sns.clustermap(tempDF, method='ward', metric='euclidean', cmap=cmap,
                    row_cluster=True, col_cluster=True, row_linkage=None, col_linkage=None,
                    row_colors=None, col_colors=None, xticklabels=True, yticklabels=True,
                    dendrogram_ratio=(0.1, 0.1), cbar_pos=(0.9, 0.02, 0.02, 0.1),
                    figsize=(15, 15), **{'center':0, 'vmin':-1, 'vmax':1})
cm.cax.set_title('Pearson\'s '+r'$r$')
hm = cm.ax_heatmap.get_position()
rd = cm.ax_row_dendrogram.get_position()
cd = cm.ax_col_dendrogram.get_position()
cm.ax_heatmap.set_position([hm.x0, hm.y0, hm.width, hm.height])
cm.ax_row_dendrogram.set_position([rd.x0+rd.width*0.5, rd.y0, rd.width*0.5, rd.height])
cm.ax_col_dendrogram.set_position([cd.x0, cd.y0, cd.width, cd.height*0.5])
plt.show()

### 1-6. Explained variance of variables with non-zero beta-coefficient

> OLS linear regression model for significance:  
> ***BMI ~ b0 + b1\*Variable + b2\*Sex + b3\*BaseAge + b4\*PCs***  
> Univariate model:  
> ***BMI ~ b0 + b1\*Variable***  
> –> Biological BMI is included for comparison.  

In [ ]:
#Female model
print('Female model:')

#Variables with non-zero beta-coefficient at least 1 of 10 models
tempL = metBMI_F_bcoefs.loc[metBMI_F_bcoefs['nZeros']!=10].index.tolist()

#Prepare dataframe for OLS regressions
tempL = [item for sublist in [tempL, covarL, ['log_BaseMetBMI']] for item in sublist]
tempDF = metDF_F[tempL]
tempDF = tempDF.rename(columns={'log_BaseMetBMI': 'MetBMI model'})

#Extract independent variables in OLS regressions (including metBMI)
tempL = tempDF.drop(columns=covarL).columns.tolist()

#Perform OLS regressions
tempL1 = []#For R2
tempL2 = []#For beta-coefficient
tempL3 = []#For SE of beta-coefficient
tempL4 = []#For p-value
tempL5 = []#For R2 in univariate model
t_start = time.time()
for var in tempL:
    tempDF_ols = tempDF[[item for sublist in [covarL, [var]] for item in sublist]]
    tempDF_ols = tempDF_ols.rename(columns={var: 'Variable'})
    #Add a constant for the intercept -> Similar to R, smf automatically add a constant

    #Univariate model
    #Fit model
    formula = 'log_BaseBMI ~ Variable'
    resOLS = smf.ols(formula, data=tempDF_ols).fit()
    #Save R2 [%]
    tempL5.append(resOLS.rsquared*100)
    
    #Full model
    #Fit model (Sex is invariant in this model and eliminated)
    formula = 'log_BaseBMI ~ Variable + BaseAge + PC1 + PC2 + PC3 + PC4 + PC5'
    resOLS = smf.ols(formula, data=tempDF_ols).fit()
    #Save R2 [%]
    tempL1.append(resOLS.rsquared*100)
    #Save beta-coefficient of the variable
    tempL2.append(resOLS.params['Variable'])
    tempL3.append(resOLS.bse['Variable'])
    #Save p-value of the variable
    tempL4.append(resOLS.pvalues['Variable'])
t_elapsed = time.time() - t_start
print('Elapsed time for ', len(tempL), 'OLS regressions (including metBMI):',
      round(t_elapsed//60), 'min', round(t_elapsed%60, 1), 'sec')

#Clean the results
tempDF = pd.DataFrame({'UnivarR2':tempL5, 'R2':tempL1, 'OLSbcoef':tempL2, 'OLSbcoefSE':tempL3, 'Pval':tempL4},
                      index=tempL)
tempDF.index.set_names('Variable', inplace=True)
##P-value adjustment by using Benjamini–Hochberg method
tempDF['AdjPval'] = multi.multipletests(tempDF['Pval'], alpha=0.05, method='fdr_bh',
                                        is_sorted=False, returnsorted=False)[1]
##Add the LASSO results
tempDF['LASSObcoef'] = metBMI_F_bcoefs.loc[metBMI_F_bcoefs.index.isin(tempL)]['MEAN']#NaN in metBMI reference
tempDF['LASSOnZeros'] = metBMI_F_bcoefs.loc[metBMI_F_bcoefs.index.isin(tempL)]['nZeros']#NaN in metBMI reference
tempDF = tempDF.sort_values(by='AdjPval', ascending=True)

#Save
fileDir = './ExportData/'
fileName = '210105_Biological-BMI-paper_LASSO-bcoef_OLS_metBMI-Female.tsv'
tempDF.to_csv(fileDir+fileName, sep='\t', index=True)

#Extact significant variables
tempDF = tempDF[tempDF['AdjPval']<0.05]
print('Variables significantly associated with BMI (FDR < 0.05):', len(tempDF)-1)#metBMI reference

#Top 30 (+ metBMI)
topX = 30+1
display(tempDF.iloc[0:topX])
##Category
tempL = []
for row_n in tempDF.index.tolist():
    if row_n=='MetBMI model':
        tempL.append('Positive association')
    else:
        if tempDF.loc[row_n, 'OLSbcoef'] > 0:
            tempL.append('Positive association')
        elif tempDF.loc[row_n, 'OLSbcoef'] < 0:
            tempL.append('Negative association')
        else:
            tempL.append('Error')
tempDF['Association'] = tempL
##Plot
tempDF = tempDF.reset_index()
sns.set(style='ticks', font='Arial', context='talk')
plt.figure(figsize=(10, 4))
p = sns.barplot(data=tempDF.iloc[0:topX], y='UnivarR2', x='Variable', hue='Association', dodge=False,
                palette={'Positive association':'tab:red', 'Negative association':'tab:blue'}, edgecolor='k')
p.set(ylim=(0, 80), yticks=np.arange(0, 80, 15))
p.grid(axis='y', linestyle='--', color='black')
sns.despine()
plt.xticks(rotation=90, horizontalalignment='center')
plt.xlabel('')
plt.ylabel('Ratio of explained variance [%]')
plt.legend(loc='upper right')
plt.show()

In [ ]:
#Male model
print('Male model:')

#Variables with non-zero beta-coefficient at least 1 of 10 models
tempL = metBMI_M_bcoefs.loc[metBMI_M_bcoefs['nZeros']!=10].index.tolist()

#Prepare dataframe for OLS regressions
tempL = [item for sublist in [tempL, covarL, ['log_BaseMetBMI']] for item in sublist]
tempDF = metDF_M[tempL]
tempDF = tempDF.rename(columns={'log_BaseMetBMI': 'MetBMI model'})

#Extract independent variables in OLS regressions (including metBMI)
tempL = tempDF.drop(columns=covarL).columns.tolist()

#Perform OLS regressions
tempL1 = []#For R2
tempL2 = []#For beta-coefficient
tempL3 = []#For SE of beta-coefficient
tempL4 = []#For p-value
tempL5 = []#For R2 in univariate model
t_start = time.time()
for var in tempL:
    tempDF_ols = tempDF[[item for sublist in [covarL, [var]] for item in sublist]]
    tempDF_ols = tempDF_ols.rename(columns={var: 'Variable'})
    #Add a constant for the intercept -> Similar to R, smf automatically add a constant

    #Univariate model
    #Fit model
    formula = 'log_BaseBMI ~ Variable'
    resOLS = smf.ols(formula, data=tempDF_ols).fit()
    #Save R2 [%]
    tempL5.append(resOLS.rsquared*100)
    
    #Full model
    #Fit model (Sex is invariant in this model and eliminated)
    formula = 'log_BaseBMI ~ Variable + BaseAge + PC1 + PC2 + PC3 + PC4 + PC5'
    resOLS = smf.ols(formula, data=tempDF_ols).fit()
    #Save R2 [%]
    tempL1.append(resOLS.rsquared*100)
    #Save beta-coefficient of the variable
    tempL2.append(resOLS.params['Variable'])
    tempL3.append(resOLS.bse['Variable'])
    #Save p-value of the variable
    tempL4.append(resOLS.pvalues['Variable'])
t_elapsed = time.time() - t_start
print('Elapsed time for ', len(tempL), 'OLS regressions (including metBMI):',
      round(t_elapsed//60), 'min', round(t_elapsed%60, 1), 'sec')

#Clean the results
tempDF = pd.DataFrame({'UnivarR2':tempL5, 'R2':tempL1, 'OLSbcoef':tempL2, 'OLSbcoefSE':tempL3, 'Pval':tempL4},
                      index=tempL)
tempDF.index.set_names('Variable', inplace=True)
##P-value adjustment by using Benjamini–Hochberg method
tempDF['AdjPval'] = multi.multipletests(tempDF['Pval'], alpha=0.05, method='fdr_bh',
                                        is_sorted=False, returnsorted=False)[1]
##Add the LASSO results
tempDF['LASSObcoef'] = metBMI_M_bcoefs.loc[metBMI_M_bcoefs.index.isin(tempL)]['MEAN']#NaN in metBMI reference
tempDF['LASSOnZeros'] = metBMI_M_bcoefs.loc[metBMI_M_bcoefs.index.isin(tempL)]['nZeros']#NaN in metBMI reference
tempDF = tempDF.sort_values(by='AdjPval', ascending=True)

#Save
fileDir = './ExportData/'
fileName = '210105_Biological-BMI-paper_LASSO-bcoef_OLS_metBMI-Male.tsv'
tempDF.to_csv(fileDir+fileName, sep='\t', index=True)

#Extact significant variables
tempDF = tempDF[tempDF['AdjPval']<0.05]
print('Variables significantly associated with BMI (FDR < 0.05):', len(tempDF)-1)#metBMI reference

#Top 30 (+ metBMI)
topX = 30+1
display(tempDF.iloc[0:topX])
##Category
tempL = []
for row_n in tempDF.index.tolist():
    if row_n=='MetBMI model':
        tempL.append('Positive association')
    else:
        if tempDF.loc[row_n, 'OLSbcoef'] > 0:
            tempL.append('Positive association')
        elif tempDF.loc[row_n, 'OLSbcoef'] < 0:
            tempL.append('Negative association')
        else:
            tempL.append('Error')
tempDF['Association'] = tempL
##Plot
tempDF = tempDF.reset_index()
sns.set(style='ticks', font='Arial', context='talk')
plt.figure(figsize=(10, 4))
p = sns.barplot(data=tempDF.iloc[0:topX], y='UnivarR2', x='Variable', hue='Association', dodge=False,
                palette={'Positive association':'tab:red', 'Negative association':'tab:blue'}, edgecolor='k')
p.set(ylim=(0, 80), yticks=np.arange(0, 80, 15))
p.grid(axis='y', linestyle='--', color='black')
sns.despine()
plt.xticks(rotation=90, horizontalalignment='center')
plt.xlabel('')
plt.ylabel('Ratio of explained variance [%]')
plt.legend(loc='upper right')
plt.show()

In [ ]:
#Both sex model
print('Both sex model:')

#Variables with non-zero beta-coefficient at least 1 of 10 models
tempL = metBMI_B_bcoefs.loc[metBMI_B_bcoefs['nZeros']!=10].index.tolist()

#Prepare dataframe for OLS regressions
tempL = [item for sublist in [tempL, covarL, ['log_BaseMetBMI']] for item in sublist]
tempDF = metDF_B[tempL]
tempDF = tempDF.rename(columns={'log_BaseMetBMI': 'MetBMI model'})

#Extract independent variables in OLS regressions (including metBMI)
tempL = tempDF.drop(columns=covarL).columns.tolist()

#Perform OLS regressions
tempL1 = []#For R2
tempL2 = []#For beta-coefficient
tempL3 = []#For SE of beta-coefficient
tempL4 = []#For p-value
tempL5 = []#For R2 in univariate model
t_start = time.time()
for var in tempL:
    tempDF_ols = tempDF[[item for sublist in [covarL, [var]] for item in sublist]]
    tempDF_ols = tempDF_ols.rename(columns={var: 'Variable'})
    #Add a constant for the intercept -> Similar to R, smf automatically add a constant

    #Univariate model
    #Fit model
    formula = 'log_BaseBMI ~ Variable'
    resOLS = smf.ols(formula, data=tempDF_ols).fit()
    #Save R2 [%]
    tempL5.append(resOLS.rsquared*100)
    
    #Full model
    #Fit model
    formula = 'log_BaseBMI ~ Variable + C(Sex) + BaseAge + PC1 + PC2 + PC3 + PC4 + PC5'
    resOLS = smf.ols(formula, data=tempDF_ols).fit()
    #Save R2 [%]
    tempL1.append(resOLS.rsquared*100)
    #Save beta-coefficient of the variable
    tempL2.append(resOLS.params['Variable'])
    tempL3.append(resOLS.bse['Variable'])
    #Save p-value of the variable
    tempL4.append(resOLS.pvalues['Variable'])
t_elapsed = time.time() - t_start
print('Elapsed time for ', len(tempL), 'OLS regressions (including metBMI):',
      round(t_elapsed//60), 'min', round(t_elapsed%60, 1), 'sec')

#Clean the results
tempDF = pd.DataFrame({'UnivarR2':tempL5, 'R2':tempL1, 'OLSbcoef':tempL2, 'OLSbcoefSE':tempL3, 'Pval':tempL4},
                      index=tempL)
tempDF.index.set_names('Variable', inplace=True)
##P-value adjustment by using Benjamini–Hochberg method
tempDF['AdjPval'] = multi.multipletests(tempDF['Pval'], alpha=0.05, method='fdr_bh',
                                        is_sorted=False, returnsorted=False)[1]
##Add the LASSO results
tempDF['LASSObcoef'] = metBMI_B_bcoefs.loc[metBMI_B_bcoefs.index.isin(tempL)]['MEAN']#NaN in metBMI reference
tempDF['LASSOnZeros'] = metBMI_B_bcoefs.loc[metBMI_B_bcoefs.index.isin(tempL)]['nZeros']#NaN in metBMI reference
tempDF = tempDF.sort_values(by='AdjPval', ascending=True)

#Save
fileDir = './ExportData/'
fileName = '210105_Biological-BMI-paper_LASSO-bcoef_OLS_metBMI-BothSex.tsv'
tempDF.to_csv(fileDir+fileName, sep='\t', index=True)

#Extact significant variables
tempDF = tempDF[tempDF['AdjPval']<0.05]
print('Variables significantly associated with BMI (FDR < 0.05):', len(tempDF)-1)#metBMI reference

#Top 30 (+ metBMI)
topX = 30+1
display(tempDF.iloc[0:topX])
##Category
tempL = []
for row_n in tempDF.index.tolist():
    if row_n=='MetBMI model':
        tempL.append('Positive association')
    else:
        if tempDF.loc[row_n, 'OLSbcoef'] > 0:
            tempL.append('Positive association')
        elif tempDF.loc[row_n, 'OLSbcoef'] < 0:
            tempL.append('Negative association')
        else:
            tempL.append('Error')
tempDF['Association'] = tempL
##Plot
tempDF = tempDF.reset_index()
sns.set(style='ticks', font='Arial', context='talk')
plt.figure(figsize=(10, 4))
p = sns.barplot(data=tempDF.iloc[0:topX], y='UnivarR2', x='Variable', hue='Association', dodge=False,
                palette={'Positive association':'tab:red', 'Negative association':'tab:blue'}, edgecolor='k')
p.set(ylim=(0, 80), yticks=np.arange(0, 80, 15))
p.grid(axis='y', linestyle='--', color='black')
sns.despine()
plt.xticks(rotation=90, horizontalalignment='center')
plt.xlabel('')
plt.ylabel('Explained variance in BMI [%]')
plt.legend(loc='upper right')
##Save
fileDir = './ExportFigures/'
fileName = '210823_Biological-BMI-paper_LASSO-bcoef_OLS_metBMI-BothSex_top30.tif'
plt.gcf().savefig(fileDir+fileName, dpi=300, bbox_inches='tight', pad_inches=0,
                  pil_kwargs={'compression':'tiff_lzw'})
plt.show()

## 2. Clinical labs

### 2-1. Prepare variables

> Variables are standardized for OLS.

In [ ]:
#Import the cleaned baseline dataframe for LASSO
fileDir = './ExportData/'
fileName = '210104_Biological-BMI-paper_RF-imputation_baseline-chemDF-with-RF-imputation.tsv'
tempDF = pd.read_csv(fileDir+fileName, sep='\t', dtype={'public_client_id': str}).set_index('public_client_id')

#Splitting dataset into sex-dependent cohorts
chemDF_F = tempDF.loc[tempDF['Sex']=='F']
chemDF_M = tempDF.loc[tempDF['Sex']=='M']
chemDF_B = tempDF#Not copy just rename
print('(Female, Male, Both): (', len(chemDF_F.index), ', ', len(chemDF_M.index), ', ', len(chemDF_B.index), ')')

In [ ]:
#Female models

#Import biological BMI
fileDir = './ExportData/'
fileName = '210105_Biological-BMI-paper_baseline-LASSO_chemBMI-Female.csv'
tempDF = pd.read_csv(fileDir+fileName, dtype={'public_client_id': str}).set_index('public_client_id')
chemDF_F['log_BaseChemBMI'] = tempDF['log_BaseChemBMI']

#Z-score standardization
tempDF = chemDF_F.drop(columns=['Sex', 'Race'])
tempL1 = tempDF.index.tolist()
tempL2 = tempDF.columns.tolist()
scaler = StandardScaler(copy=True, with_mean=True, with_std=True)
tempA = scaler.fit_transform(tempDF)#Column direction
tempDF = pd.DataFrame(data=tempA, index=tempL1, columns=tempL2)
tempDF['Sex'] = chemDF_F['Sex']

#Update
chemDF_F = tempDF
display(chemDF_F.describe(include='all'))

In [ ]:
#Male model

#Import biological BMI
fileDir = './ExportData/'
fileName = '210105_Biological-BMI-paper_baseline-LASSO_chemBMI-Male.csv'
tempDF = pd.read_csv(fileDir+fileName, dtype={'public_client_id': str}).set_index('public_client_id')
chemDF_M['log_BaseChemBMI'] = tempDF['log_BaseChemBMI']

#Z-score standardization
tempDF = chemDF_M.drop(columns=['Sex', 'Race'])
tempL1 = tempDF.index.tolist()
tempL2 = tempDF.columns.tolist()
scaler = StandardScaler(copy=True, with_mean=True, with_std=True)
tempDF = scaler.fit_transform(tempDF)#Column direction
tempDF = pd.DataFrame(data=tempDF, index=tempL1, columns=tempL2)
tempDF['Sex'] = chemDF_M['Sex']

#Update
chemDF_M = tempDF
display(chemDF_M.describe(include='all'))

In [ ]:
#Both sex model

#Import biological BMI
fileDir = './ExportData/'
fileName = '210105_Biological-BMI-paper_baseline-LASSO_chemBMI-BothSex.csv'
tempDF = pd.read_csv(fileDir+fileName, dtype={'public_client_id': str}).set_index('public_client_id')
chemDF_B['log_BaseChemBMI'] = tempDF['log_BaseChemBMI']

#Z-score standardization
tempDF = chemDF_B.drop(columns=['Sex', 'Race'])
tempL1 = tempDF.index.tolist()
tempL2 = tempDF.columns.tolist()
scaler = StandardScaler(copy=True, with_mean=True, with_std=True)
tempDF = scaler.fit_transform(tempDF)#Column direction
tempDF = pd.DataFrame(data=tempDF, index=tempL1, columns=tempL2)
tempDF['Sex'] = chemDF_B['Sex']

#Update
chemDF_B = tempDF
display(chemDF_B.describe(include='all'))

### 2-2. Correlation b/w all pairwise variables

In [ ]:
#Female and male cohorts
tempL = [item for sublist in [covarL, ['log_BaseChemBMI']] for item in sublist]

print('Female cohort:')
#Compute correlation matrix
tempDF1 = chemDF_F.drop(columns=tempL)
tempDF1 = tempDF1.corr(method='pearson')
print('• Combinations:', int(len(tempDF1)*(len(tempDF1)-1)/2))

#Extract lower triangle matrix
tempDF1 = tempDF1.where(np.tril(np.ones(tempDF1.shape), k=-1).astype(np.bool), other=np.nan)
tempDF1.index.set_names('Variable1', inplace=True)
tempDF1 = tempDF1.reset_index().melt(var_name='Variable2', value_name='Pearson_r', id_vars=['Variable1'])
tempDF1 = tempDF1.dropna()
print('• nrows:', len(tempDF1))
print('• |Pearson\'s r| > 0.8:', len(tempDF1.loc[abs(tempDF1['Pearson_r'])>0.8]))
display(tempDF1.loc[abs(tempDF1['Pearson_r'])>0.8])

print('Male cohort:')
#Compute correlation matrix
tempDF2 = chemDF_M.drop(columns=tempL)
tempDF2 = tempDF2.corr(method='pearson')
print('• Combinations:', int(len(tempDF2)*(len(tempDF2)-1)/2))

#Extract lower triangle matrix
tempDF2 = tempDF2.where(np.tril(np.ones(tempDF2.shape), k=-1).astype(np.bool), other=np.nan)
tempDF2.index.set_names('Variable1', inplace=True)
tempDF2 = tempDF2.reset_index().melt(var_name='Variable2', value_name='Pearson_r', id_vars=['Variable1'])
tempDF2 = tempDF2.dropna()
print('• nrows:', len(tempDF2))
print('• |Pearson\'s r| > 0.8:', len(tempDF2.loc[abs(tempDF2['Pearson_r'])>0.8]))
display(tempDF2.loc[abs(tempDF2['Pearson_r'])>0.8])

#Distribution
sns.set(style='ticks', font='Arial', context='notebook')
plt.figure(figsize=(4, 3))
sns.distplot(tempDF1['Pearson_r'], color='r', label='Female').set(xlim=(-1, 1), xticks=np.arange(-1, 1.1, 0.5))
sns.distplot(tempDF2['Pearson_r'], color='b', label='Male').set(xlim=(-1, 1), xticks=np.arange(-1, 1.1, 0.5))
sns.despine()
plt.xlabel('Pearson\'s '+r'$r$')
plt.ylabel('Density')
plt.legend(bbox_to_anchor=(1, 0.5), loc='center left', borderaxespad=1)
plt.show()

In [ ]:
#Both sex cohort
tempL = [item for sublist in [covarL, ['log_BaseChemBMI']] for item in sublist]

print('Both sex cohort:')
#Compute correlation matrix
tempDF = chemDF_B.drop(columns=tempL)
tempDF = tempDF.corr(method='pearson')
print('• Combinations:', int(len(tempDF)*(len(tempDF)-1)/2))

#Extract lower triangle matrix
tempDF = tempDF.where(np.tril(np.ones(tempDF.shape), k=-1).astype(np.bool), other=np.nan)
tempDF.index.set_names('Variable1', inplace=True)
tempDF = tempDF.reset_index().melt(var_name='Variable2', value_name='Pearson_r', id_vars=['Variable1'])
tempDF = tempDF.dropna()
print('• nrows:', len(tempDF))
print('• |Pearson\'s r| > 0.8:', len(tempDF.loc[abs(tempDF['Pearson_r'])>0.8]))
display(tempDF.loc[abs(tempDF['Pearson_r'])>0.8])

#Distribution
sns.set(style='ticks', font='Arial', context='talk')
plt.figure(figsize=(4, 3))
sns.distplot(tempDF['Pearson_r'], color='g').set(xlim=(-1, 1), xticks=np.arange(-1, 1.1, 0.5))
sns.despine()
plt.ylabel('Density')
plt.xlabel('Pearson\'s '+r'$r$')
plt.axvline(x=tempDF['Pearson_r'].max(), **{'linestyle':'--', 'color':'g'})
plt.axvline(x=tempDF['Pearson_r'].min(), **{'linestyle':'--', 'color':'g'})
plt.show()

### 2-3. Beta-coefficients of LASSO models

In [ ]:
#Female model
#Import the cleaned beta-coefficients of LASSO
fileDir = './ExportData/'
fileName = '210105_Biological-BMI-paper_baseline-LASSO_chemBMI-Female_LASSO-bcoefs.tsv'
chemBMI_F_bcoefs = pd.read_csv(fileDir+fileName, sep='\t').set_index('Variable')
print('Variables:', len(chemBMI_F_bcoefs))

#Variables with non-zero beta-coefficient
tempDF = chemBMI_F_bcoefs.loc[chemBMI_F_bcoefs['nZeros']!=10]
print('Variables with non-zero beta-coefficient:', len(tempDF),
      '(', len(tempDF)/len(chemBMI_F_bcoefs)*100, '%)')

#Variables with non-zero beta-coefficient in all 10 models
tempDF = chemBMI_F_bcoefs.loc[chemBMI_F_bcoefs['nZeros']==0]
tempDF = tempDF.sort_values(by='MEAN', ascending=False)
print('Variables with non-zero beta-coefficient in all 10 models:', len(tempDF),
      '(', len(tempDF)/len(chemBMI_F_bcoefs)*100, '%)')
display(tempDF)

In [ ]:
#Male model
#Import the cleaned beta-coefficients of LASSO
fileDir = './ExportData/'
fileName = '210105_Biological-BMI-paper_baseline-LASSO_chemBMI-Male_LASSO-bcoefs.tsv'
chemBMI_M_bcoefs = pd.read_csv(fileDir+fileName, sep='\t').set_index('Variable')
print('Variables:', len(chemBMI_M_bcoefs))

#Variables with non-zero beta-coefficient
tempDF = chemBMI_M_bcoefs.loc[chemBMI_M_bcoefs['nZeros']!=10]
print('Variables with non-zero beta-coefficient:', len(tempDF),
      '(', len(tempDF)/len(chemBMI_M_bcoefs)*100, '%)')

#Variables with non-zero beta-coefficient in all 10 models
tempDF = chemBMI_M_bcoefs.loc[chemBMI_M_bcoefs['nZeros']==0]
tempDF = tempDF.sort_values(by='MEAN', ascending=False)
print('Variables with non-zero beta-coefficient in all 10 models:', len(tempDF),
      '(', len(tempDF)/len(chemBMI_M_bcoefs)*100, '%)')
display(tempDF)

In [ ]:
#Both sex model
#Import the cleaned beta-coefficients of LASSO
fileDir = './ExportData/'
fileName = '210105_Biological-BMI-paper_baseline-LASSO_chemBMI-BothSex_LASSO-bcoefs.tsv'
chemBMI_B_bcoefs = pd.read_csv(fileDir+fileName, sep='\t').set_index('Variable')
print('Variables:', len(chemBMI_B_bcoefs))

#Variables with non-zero beta-coefficient
tempDF = chemBMI_B_bcoefs.loc[chemBMI_B_bcoefs['nZeros']!=10]
print('Variables with non-zero beta-coefficient:', len(tempDF),
      '(', len(tempDF)/len(chemBMI_B_bcoefs)*100, '%)')

#Variables with non-zero beta-coefficient in all 10 models
tempDF = chemBMI_B_bcoefs.loc[chemBMI_B_bcoefs['nZeros']==0]
tempDF = tempDF.sort_values(by='MEAN', ascending=False)
print('Variables with non-zero beta-coefficient in all 10 models:', len(tempDF),
      '(', len(tempDF)/len(chemBMI_B_bcoefs)*100, '%)')
display(tempDF)

### 2-4. Variables with non-zero beta-coefficient in all 10 models

In [ ]:
#Variables with non-zero beta-coefficient in all 10 models
tempS1 = set(chemBMI_F_bcoefs.loc[chemBMI_F_bcoefs['nZeros']==0].index)
tempS2 = set(chemBMI_M_bcoefs.loc[chemBMI_M_bcoefs['nZeros']==0].index)
tempS3 = set(chemBMI_B_bcoefs.loc[chemBMI_B_bcoefs['nZeros']==0].index)

#Common the robust variables
var_111 = tempS1 & tempS2 & tempS3
print('Common variables with non-zero beta-coefficient in all 10 models:\n', var_111)
var_110 = (tempS1 & tempS2) - var_111
var_101 = (tempS1 & tempS3) - var_111
var_011 = (tempS2 & tempS3) - var_111
var_100 = tempS1 - (var_110 | var_101 | var_111)
var_010 = tempS2 - (var_110 | var_011 | var_111)
var_001 = tempS3 - (var_101 | var_011 | var_111)

#The number of each subset
var_subset = [len(var_100), len(var_010), len(var_110), len(var_001),
              len(var_101), len(var_011), len(var_111)]
print('Each subset:', var_subset)

#Venn diagram
sns.set(font='Arial', context='talk')
venn3(subsets=var_subset, set_labels=('Female', 'Male', 'Both sex'), set_colors=('r', 'b', 'g'), alpha=0.4)
venn3_circles(subsets=var_subset)
plt.title('Robust variables\n—Clinical labs—', fontdict={'fontsize':24})
plt.show()

In [ ]:
#Variables with non-zero beta-coefficient in all 10 models
tempS1 = set(chemBMI_F_bcoefs.loc[chemBMI_F_bcoefs['nZeros']==0].index)
tempS2 = set(chemBMI_M_bcoefs.loc[chemBMI_M_bcoefs['nZeros']==0].index)
tempS3 = set(chemBMI_B_bcoefs.loc[chemBMI_B_bcoefs['nZeros']==0].index)

#Union of the robust variables
tempL = list(tempS1 | tempS2 | tempS3)
print('Union of the variables with non-zero beta-coefficient in all 10 models:', len(tempL))

#Merge (use original bcoefs to rescue ones with zeros in any 10 models)
tempDF1 = chemBMI_F_bcoefs.loc[chemBMI_F_bcoefs.index.isin(tempL)].drop(columns=['MEAN', 'SD', 'nZeros'])
tempDF2 = chemBMI_M_bcoefs.loc[chemBMI_M_bcoefs.index.isin(tempL)].drop(columns=['MEAN', 'SD', 'nZeros'])
tempDF3 = chemBMI_B_bcoefs.loc[chemBMI_B_bcoefs.index.isin(tempL)].drop(columns=['MEAN', 'SD', 'nZeros'])
tempDF = pd.concat([tempDF1, tempDF2, tempDF3], axis=0)
tempDF['Model'] = np.repeat(['Female', 'Male', 'Both sex'], [len(tempDF1), len(tempDF2), len(tempDF3)])

#tidyr::gather in R
tempDF1 = tempDF.reset_index().melt(var_name='LASSO', value_name='bcoef', id_vars=['Variable', 'Model'])

#Order for plot
tempDF2 = tempDF1.loc[tempDF1['Model']=='Both sex'].groupby('Variable', as_index=True).agg({'bcoef': np.mean})
tempDF2 = tempDF2.sort_values(by='bcoef', ascending=False)

#Plot
sns.set(style='ticks', font='Arial', context='talk')
sns.catplot(data=tempDF1, y='Variable', order=tempDF2.index.tolist(), x='bcoef', kind='box',
                hue='Model', palette='Set1', dodge=True, height=9, aspect=1.1,
                showfliers=True, flierprops={'marker':'o', 'markerfacecolor':'gray', 'alpha':0.4})
plt.xlabel(r'$\beta$'+'-coefficient in LASSO models')
plt.ylabel('')
tempA = np.arange(-0.04, 0.061, 0.02)
for coord in tempA:
    plt.axvline(x=coord, **{'linestyle':'--', 'color':'gray'})
for row_i in range(len(tempDF2)):
    if row_i%2 == 0:
        plt.axhspan(ymin=row_i-0.5, ymax=row_i+0.5, facecolor='g', alpha=0.1)
plt.show()

In [ ]:
#Both sex model

#Variables with non-zero beta-coefficient in all 10 models
tempDF = chemBMI_B_bcoefs.loc[chemBMI_B_bcoefs['nZeros']==0].sort_values(by='MEAN', ascending=False)
tempDF = tempDF.drop(columns=['MEAN', 'SD', 'nZeros'])
print('Variables with non-zero beta-coefficient in all 10 models:', len(tempDF))

#tidyr::gather in R
tempDF1 = tempDF.reset_index().melt(var_name='LASSO', value_name='bcoef', id_vars=['Variable'])

#Plot
sns.set(style='ticks', font='Arial', context='talk')
plt.figure(figsize=(5, 7))
p = sns.boxplot(data=tempDF1, y='Variable', x='bcoef', color='g', dodge=False,
                showfliers=True, flierprops={'marker':'o', 'markerfacecolor':'gray', 'alpha':0.4},
                showcaps=True, notch=False)
p.set(xlim=(-0.046, 0.056), xticks=np.arange(-0.04, 0.06, 0.02))
p.grid(axis='x', linestyle='--', color='black')
sns.despine()
plt.ylabel('')
plt.xlabel(r'$\beta$'+'-coefficient in LASSO model')
for row_i in range(len(tempDF)):
    if row_i%2 == 0:
        plt.axhspan(ymin=row_i-0.5, ymax=row_i+0.5, facecolor='g', alpha=0.2, zorder=0)
##Save
fileDir = './ExportFigures/'
fileName = '210823_Biological-BMI-paper_LASSO-bcoef_chemBMI-BothSex-bcoef_non-zero-in-all.tif'
plt.gcf().savefig(fileDir+fileName, dpi=300, bbox_inches='tight', pad_inches=0,
                  pil_kwargs={'compression':'tiff_lzw'})
plt.show()

### 2-5. Correlation b/w pairwise variables with non-zero beta coefficient in all 10 models

In [ ]:
#Both sex model
print('Both sex model:')

#Variables with non-zero beta-coefficient in all 10 models
tempL = chemBMI_B_bcoefs.loc[chemBMI_B_bcoefs['nZeros']==0].index.tolist()

#Compute correlation matrix
tempDF = chemDF_B[tempL]
tempDF = tempDF.corr(method='pearson')
print('• Combinations:', int(len(tempDF)*(len(tempDF)-1)/2))

#Extract lower triangle matrix
tempDF = tempDF.where(np.tril(np.ones(tempDF.shape), k=-1).astype(np.bool), other=np.nan)
tempDF.index.set_names('Variable1', inplace=True)
tempDF = tempDF.reset_index().melt(var_name='Variable2', value_name='Pearson_r', id_vars=['Variable1'])
tempDF = tempDF.dropna()
print('• nrows:', len(tempDF))
print('• |Pearson\'s r| > 0.8:', len(tempDF.loc[abs(tempDF['Pearson_r'])>0.8]))
display(tempDF.loc[abs(tempDF['Pearson_r'])>0.8])

#Distribution
sns.set(style='ticks', font='Arial', context='talk')
plt.figure(figsize=(4, 3))
sns.distplot(tempDF['Pearson_r'], color='g').set(xlim=(-1, 1), xticks=np.arange(-1, 1.1, 0.5))
sns.despine()
plt.ylabel('Density')
plt.xlabel('Pearson\'s '+r'$r$')
plt.axvline(x=tempDF['Pearson_r'].max(), **{'linestyle':'--', 'color':'g'})
plt.axvline(x=tempDF['Pearson_r'].min(), **{'linestyle':'--', 'color':'g'})
plt.show()

print('Variables with non-zero beta-coefficient in all 10 models:', len(tempL))
#Clustermap
tempDF = chemDF_B[tempL]
tempDF = tempDF.corr(method='pearson')
sns.set(style='ticks', font='Arial', context='notebook')
cmap = sns.diverging_palette(220, 20, as_cmap=True)
cm = sns.clustermap(tempDF, method='ward', metric='euclidean', cmap=cmap,
                    row_cluster=True, col_cluster=True, row_linkage=None, col_linkage=None,
                    row_colors=None, col_colors=None, xticklabels=True, yticklabels=True,
                    dendrogram_ratio=(0.15, 0.15), cbar_pos=(0.9, 0.02, 0.02, 0.1),
                    figsize=(6.5, 6.5), **{'center':0, 'vmin':-1, 'vmax':1})
cm.cax.set_title('Pearson\'s '+r'$r$')
hm = cm.ax_heatmap.get_position()
rd = cm.ax_row_dendrogram.get_position()
cd = cm.ax_col_dendrogram.get_position()
cm.ax_heatmap.set_position([hm.x0+rd.width, hm.y0, hm.width, hm.height])
cm.ax_row_dendrogram.set_position([rd.x0+rd.width*0.5+rd.width, rd.y0, rd.width*0.5, rd.height])
cm.ax_col_dendrogram.set_position([cd.x0+rd.width, cd.y0, cd.width, cd.height*0.5])
plt.show()

### 2-6. Explained variance of variables with non-zero beta-coefficient

> OLS linear regression model for significance:  
> ***BMI ~ b0 + b1\*Variable + b2\*Sex + b3\*BaseAge + b4\*PCs***  
> Univariate model:  
> ***BMI ~ b0 + b1\*Variable***  
> –> Biological BMI is included for comparison.  

In [ ]:
#Female model
print('Female model:')

#Variables with non-zero beta-coefficient at least 1 of 10 models
tempL = chemBMI_F_bcoefs.loc[chemBMI_F_bcoefs['nZeros']!=10].index.tolist()

#Prepare dataframe for OLS regressions
tempL = [item for sublist in [tempL, covarL, ['log_BaseChemBMI']] for item in sublist]
tempDF = chemDF_F[tempL]
tempDF = tempDF.rename(columns={'log_BaseChemBMI': 'ChemBMI model'})

#Extract independent variables in OLS regressions (including chemBMI)
tempL = tempDF.drop(columns=covarL).columns.tolist()

#Perform OLS regressions
tempL1 = []#For R2
tempL2 = []#For beta-coefficient
tempL3 = []#For SE of beta-coefficient
tempL4 = []#For p-value
tempL5 = []#For R2 in univariate model
t_start = time.time()
for var in tempL:
    tempDF_ols = tempDF[[item for sublist in [covarL, [var]] for item in sublist]]
    tempDF_ols = tempDF_ols.rename(columns={var: 'Variable'})
    #Add a constant for the intercept -> Similar to R, smf automatically add a constant

    #Univariate model
    #Fit model
    formula = 'log_BaseBMI ~ Variable'
    resOLS = smf.ols(formula, data=tempDF_ols).fit()
    #Save R2 [%]
    tempL5.append(resOLS.rsquared*100)
    
    #Full model
    #Fit model (Sex is invariant in this model and eliminated)
    formula = 'log_BaseBMI ~ Variable + BaseAge + PC1 + PC2 + PC3 + PC4 + PC5'
    resOLS = smf.ols(formula, data=tempDF_ols).fit()
    #Save R2 [%]
    tempL1.append(resOLS.rsquared*100)
    #Save beta-coefficient of the variable
    tempL2.append(resOLS.params['Variable'])
    tempL3.append(resOLS.bse['Variable'])
    #Save p-value of the variable
    tempL4.append(resOLS.pvalues['Variable'])
t_elapsed = time.time() - t_start
print('Elapsed time for ', len(tempL), 'OLS regressions (including chemBMI):',
      round(t_elapsed//60), 'min', round(t_elapsed%60, 1), 'sec')

#Clean the results
tempDF = pd.DataFrame({'UnivarR2':tempL5, 'R2':tempL1, 'OLSbcoef':tempL2, 'OLSbcoefSE':tempL3, 'Pval':tempL4},
                      index=tempL)
tempDF.index.set_names('Variable', inplace=True)
##P-value adjustment by using Benjamini–Hochberg method
tempDF['AdjPval'] = multi.multipletests(tempDF['Pval'], alpha=0.05, method='fdr_bh',
                                        is_sorted=False, returnsorted=False)[1]
##Add the LASSO results
tempDF['LASSObcoef'] = chemBMI_F_bcoefs.loc[chemBMI_F_bcoefs.index.isin(tempL)]['MEAN']#NaN in chemBMI reference
tempDF['LASSOnZeros'] = chemBMI_F_bcoefs.loc[chemBMI_F_bcoefs.index.isin(tempL)]['nZeros']#NaN in chemBMI reference
tempDF = tempDF.sort_values(by='AdjPval', ascending=True)

#Save
fileDir = './ExportData/'
fileName = '210105_Biological-BMI-paper_LASSO-bcoef_OLS_chemBMI-Female.tsv'
tempDF.to_csv(fileDir+fileName, sep='\t', index=True)

#Extact significant variables
tempDF = tempDF[tempDF['AdjPval']<0.05]
print('Variables significantly associated with BMI (FDR < 0.05):', len(tempDF)-1)#chemBMI reference

#Top 30 (+ chemBMI)
topX = 30+1
display(tempDF.iloc[0:topX])
##Category
tempL = []
for row_n in tempDF.index.tolist():
    if row_n=='ChemBMI model':
        tempL.append('Positive association')
    else:
        if tempDF.loc[row_n, 'OLSbcoef'] > 0:
            tempL.append('Positive association')
        elif tempDF.loc[row_n, 'OLSbcoef'] < 0:
            tempL.append('Negative association')
        else:
            tempL.append('Error')
tempDF['Association'] = tempL
##Plot
tempDF = tempDF.reset_index()
sns.set(style='ticks', font='Arial', context='talk')
plt.figure(figsize=(10, 4))
p = sns.barplot(data=tempDF.iloc[0:topX], y='UnivarR2', x='Variable', hue='Association', dodge=False,
                palette={'Positive association':'tab:red', 'Negative association':'tab:blue'}, edgecolor='k')
p.set(ylim=(0, 80), yticks=np.arange(0, 80, 15))
p.grid(axis='y', linestyle='--', color='black')
sns.despine()
plt.xticks(rotation=90, horizontalalignment='center')
plt.xlabel('')
plt.ylabel('Ratio of explained variance [%]')
plt.legend(loc='upper right')
plt.show()

In [ ]:
#Male model
print('Male model:')

#Variables with non-zero beta-coefficient at least 1 of 10 models
tempL = chemBMI_M_bcoefs.loc[chemBMI_M_bcoefs['nZeros']!=10].index.tolist()

#Prepare dataframe for OLS regressions
tempL = [item for sublist in [tempL, covarL, ['log_BaseChemBMI']] for item in sublist]
tempDF = chemDF_M[tempL]
tempDF = tempDF.rename(columns={'log_BaseChemBMI': 'ChemBMI model'})

#Extract independent variables in OLS regressions (including chemBMI)
tempL = tempDF.drop(columns=covarL).columns.tolist()

#Perform OLS regressions
tempL1 = []#For R2
tempL2 = []#For beta-coefficient
tempL3 = []#For SE of beta-coefficient
tempL4 = []#For p-value
tempL5 = []#For R2 in univariate model
t_start = time.time()
for var in tempL:
    tempDF_ols = tempDF[[item for sublist in [covarL, [var]] for item in sublist]]
    tempDF_ols = tempDF_ols.rename(columns={var: 'Variable'})
    #Add a constant for the intercept -> Similar to R, smf automatically add a constant

    #Univariate model
    #Fit model
    formula = 'log_BaseBMI ~ Variable'
    resOLS = smf.ols(formula, data=tempDF_ols).fit()
    #Save R2 [%]
    tempL5.append(resOLS.rsquared*100)
    
    #Full model
    #Fit model (Sex is invariant in this model and eliminated)
    formula = 'log_BaseBMI ~ Variable + BaseAge + PC1 + PC2 + PC3 + PC4 + PC5'
    resOLS = smf.ols(formula, data=tempDF_ols).fit()
    #Save R2 [%]
    tempL1.append(resOLS.rsquared*100)
    #Save beta-coefficient of the variable
    tempL2.append(resOLS.params['Variable'])
    tempL3.append(resOLS.bse['Variable'])
    #Save p-value of the variable
    tempL4.append(resOLS.pvalues['Variable'])
t_elapsed = time.time() - t_start
print('Elapsed time for ', len(tempL), 'OLS regressions (including chemBMI):',
      round(t_elapsed//60), 'min', round(t_elapsed%60, 1), 'sec')

#Clean the results
tempDF = pd.DataFrame({'UnivarR2':tempL5, 'R2':tempL1, 'OLSbcoef':tempL2, 'OLSbcoefSE':tempL3, 'Pval':tempL4},
                      index=tempL)
tempDF.index.set_names('Variable', inplace=True)
##P-value adjustment by using Benjamini–Hochberg method
tempDF['AdjPval'] = multi.multipletests(tempDF['Pval'], alpha=0.05, method='fdr_bh',
                                        is_sorted=False, returnsorted=False)[1]
##Add the LASSO results
tempDF['LASSObcoef'] = chemBMI_M_bcoefs.loc[chemBMI_M_bcoefs.index.isin(tempL)]['MEAN']#NaN in chemBMI reference
tempDF['LASSOnZeros'] = chemBMI_M_bcoefs.loc[chemBMI_M_bcoefs.index.isin(tempL)]['nZeros']#NaN in chemBMI reference
tempDF = tempDF.sort_values(by='AdjPval', ascending=True)

#Save
fileDir = './ExportData/'
fileName = '210105_Biological-BMI-paper_LASSO-bcoef_OLS_chemBMI-Male.tsv'
tempDF.to_csv(fileDir+fileName, sep='\t', index=True)

#Extact significant variables
tempDF = tempDF[tempDF['AdjPval']<0.05]
print('Variables significantly associated with BMI (FDR < 0.05):', len(tempDF)-1)#chemBMI reference

#Top 30 (+ chemBMI)
topX = 30+1
display(tempDF.iloc[0:topX])
##Category
tempL = []
for row_n in tempDF.index.tolist():
    if row_n=='ChemBMI model':
        tempL.append('Positive association')
    else:
        if tempDF.loc[row_n, 'OLSbcoef'] > 0:
            tempL.append('Positive association')
        elif tempDF.loc[row_n, 'OLSbcoef'] < 0:
            tempL.append('Negative association')
        else:
            tempL.append('Error')
tempDF['Association'] = tempL
##Plot
tempDF = tempDF.reset_index()
sns.set(style='ticks', font='Arial', context='talk')
plt.figure(figsize=(10, 4))
p = sns.barplot(data=tempDF.iloc[0:topX], y='UnivarR2', x='Variable', hue='Association', dodge=False,
                palette={'Positive association':'tab:red', 'Negative association':'tab:blue'}, edgecolor='k')
p.set(ylim=(0, 80), yticks=np.arange(0, 80, 15))
p.grid(axis='y', linestyle='--', color='black')
sns.despine()
plt.xticks(rotation=90, horizontalalignment='center')
plt.xlabel('')
plt.ylabel('Ratio of explained variance [%]')
plt.legend(loc='upper right')
plt.show()

In [ ]:
#Both sex model
print('Both sex model:')

#Variables with non-zero beta-coefficient at least 1 of 10 models
tempL = chemBMI_B_bcoefs.loc[chemBMI_B_bcoefs['nZeros']!=10].index.tolist()

#Prepare dataframe for OLS regressions
tempL = [item for sublist in [tempL, covarL, ['log_BaseChemBMI']] for item in sublist]
tempDF = chemDF_B[tempL]
tempDF = tempDF.rename(columns={'log_BaseChemBMI': 'ChemBMI model'})

#Extract independent variables in OLS regressions (including chemBMI)
tempL = tempDF.drop(columns=covarL).columns.tolist()

#Perform OLS regressions
tempL1 = []#For R2
tempL2 = []#For beta-coefficient
tempL3 = []#For SE of beta-coefficient
tempL4 = []#For p-value
tempL5 = []#For R2 in univariate model
t_start = time.time()
for var in tempL:
    tempDF_ols = tempDF[[item for sublist in [covarL, [var]] for item in sublist]]
    tempDF_ols = tempDF_ols.rename(columns={var: 'Variable'})
    #Add a constant for the intercept -> Similar to R, smf automatically add a constant

    #Univariate model
    #Fit model
    formula = 'log_BaseBMI ~ Variable'
    resOLS = smf.ols(formula, data=tempDF_ols).fit()
    #Save R2 [%]
    tempL5.append(resOLS.rsquared*100)
    
    #Full model
    #Fit model
    formula = 'log_BaseBMI ~ Variable + C(Sex) + BaseAge + PC1 + PC2 + PC3 + PC4 + PC5'
    resOLS = smf.ols(formula, data=tempDF_ols).fit()
    #Save R2 [%]
    tempL1.append(resOLS.rsquared*100)
    #Save beta-coefficient of the variable
    tempL2.append(resOLS.params['Variable'])
    tempL3.append(resOLS.bse['Variable'])
    #Save p-value of the variable
    tempL4.append(resOLS.pvalues['Variable'])
t_elapsed = time.time() - t_start
print('Elapsed time for ', len(tempL), 'OLS regressions (including chemBMI):',
      round(t_elapsed//60), 'min', round(t_elapsed%60, 1), 'sec')

#Clean the results
tempDF = pd.DataFrame({'UnivarR2':tempL5, 'R2':tempL1, 'OLSbcoef':tempL2, 'OLSbcoefSE':tempL3, 'Pval':tempL4},
                      index=tempL)
tempDF.index.set_names('Variable', inplace=True)
##P-value adjustment by using Benjamini–Hochberg method
tempDF['AdjPval'] = multi.multipletests(tempDF['Pval'], alpha=0.05, method='fdr_bh',
                                        is_sorted=False, returnsorted=False)[1]
##Add the LASSO results
tempDF['LASSObcoef'] = chemBMI_B_bcoefs.loc[chemBMI_B_bcoefs.index.isin(tempL)]['MEAN']#NaN in chemBMI reference
tempDF['LASSOnZeros'] = chemBMI_B_bcoefs.loc[chemBMI_B_bcoefs.index.isin(tempL)]['nZeros']#NaN in chemBMI reference
tempDF = tempDF.sort_values(by='AdjPval', ascending=True)

#Save
fileDir = './ExportData/'
fileName = '210105_Biological-BMI-paper_LASSO-bcoef_OLS_chemBMI-BothSex.tsv'
tempDF.to_csv(fileDir+fileName, sep='\t', index=True)

#Extact significant variables
tempDF = tempDF[tempDF['AdjPval']<0.05]
print('Variables significantly associated with BMI (FDR < 0.05):', len(tempDF)-1)#chemBMI reference

#Top 30 (+ chemBMI)
topX = 30+1
display(tempDF.iloc[0:topX])
##Category
tempL = []
for row_n in tempDF.index.tolist():
    if row_n=='ChemBMI model':
        tempL.append('Positive association')
    else:
        if tempDF.loc[row_n, 'OLSbcoef'] > 0:
            tempL.append('Positive association')
        elif tempDF.loc[row_n, 'OLSbcoef'] < 0:
            tempL.append('Negative association')
        else:
            tempL.append('Error')
tempDF['Association'] = tempL
##Plot
tempDF = tempDF.reset_index()
sns.set(style='ticks', font='Arial', context='talk')
plt.figure(figsize=(10, 4))
p = sns.barplot(data=tempDF.iloc[0:topX], y='UnivarR2', x='Variable', hue='Association', dodge=False,
                palette={'Positive association':'tab:red', 'Negative association':'tab:blue'}, edgecolor='k')
p.set(ylim=(0, 80), yticks=np.arange(0, 80, 15))
p.grid(axis='y', linestyle='--', color='black')
sns.despine()
plt.xticks(rotation=90, horizontalalignment='center')
plt.xlabel('')
plt.ylabel('Explained variance in BMI [%]')
plt.legend(loc='upper right')
##Save
fileDir = './ExportFigures/'
fileName = '210823_Biological-BMI-paper_LASSO-bcoef_OLS_chemBMI-BothSex_top30.tif'
plt.gcf().savefig(fileDir+fileName, dpi=300, bbox_inches='tight', pad_inches=0,
                  pil_kwargs={'compression':'tiff_lzw'})
plt.show()

## 3. Proteomics

### 3-1. Prepare variables

> Variables are standardized for OLS.

In [ ]:
#Import the cleaned baseline dataframe for LASSO
fileDir = './ExportData/'
fileName = '210104_Biological-BMI-paper_RF-imputation_baseline-protDF-with-RF-imputation.tsv'
tempDF = pd.read_csv(fileDir+fileName, sep='\t', dtype={'public_client_id': str}).set_index('public_client_id')

#Splitting dataset into sex-dependent cohorts
protDF_F = tempDF.loc[tempDF['Sex']=='F']
protDF_M = tempDF.loc[tempDF['Sex']=='M']
protDF_B = tempDF#Not copy just rename
print('(Female, Male, Both): (', len(protDF_F.index), ', ', len(protDF_M.index), ', ', len(protDF_B.index), ')')

In [ ]:
#Female models

#Import biological BMI
fileDir = './ExportData/'
fileName = '210105_Biological-BMI-paper_baseline-LASSO_protBMI-Female.csv'
tempDF = pd.read_csv(fileDir+fileName, dtype={'public_client_id': str}).set_index('public_client_id')
protDF_F['log_BaseProtBMI'] = tempDF['log_BaseProtBMI']

#Z-score standardization
tempDF = protDF_F.drop(columns=['Sex', 'Race'])
tempL1 = tempDF.index.tolist()
tempL2 = tempDF.columns.tolist()
scaler = StandardScaler(copy=True, with_mean=True, with_std=True)
tempA = scaler.fit_transform(tempDF)#Column direction
tempDF = pd.DataFrame(data=tempA, index=tempL1, columns=tempL2)
tempDF['Sex'] = protDF_F['Sex']

#Update
protDF_F = tempDF
display(protDF_F.describe(include='all'))

In [ ]:
#Male model

#Import biological BMI
fileDir = './ExportData/'
fileName = '210105_Biological-BMI-paper_baseline-LASSO_protBMI-Male.csv'
tempDF = pd.read_csv(fileDir+fileName, dtype={'public_client_id': str}).set_index('public_client_id')
protDF_M['log_BaseProtBMI'] = tempDF['log_BaseProtBMI']

#Z-score standardization
tempDF = protDF_M.drop(columns=['Sex', 'Race'])
tempL1 = tempDF.index.tolist()
tempL2 = tempDF.columns.tolist()
scaler = StandardScaler(copy=True, with_mean=True, with_std=True)
tempA = scaler.fit_transform(tempDF)#Column direction
tempDF = pd.DataFrame(data=tempA, index=tempL1, columns=tempL2)
tempDF['Sex'] = protDF_M['Sex']

#Update
protDF_M = tempDF
display(protDF_M.describe(include='all'))

In [ ]:
#Both sex model

#Import biological BMI
fileDir = './ExportData/'
fileName = '210105_Biological-BMI-paper_baseline-LASSO_protBMI-BothSex.csv'
tempDF = pd.read_csv(fileDir+fileName, dtype={'public_client_id': str}).set_index('public_client_id')
protDF_B['log_BaseProtBMI'] = tempDF['log_BaseProtBMI']

#Z-score standardization
tempDF = protDF_B.drop(columns=['Sex', 'Race'])
tempL1 = tempDF.index.tolist()
tempL2 = tempDF.columns.tolist()
scaler = StandardScaler(copy=True, with_mean=True, with_std=True)
tempA = scaler.fit_transform(tempDF)#Column direction
tempDF = pd.DataFrame(data=tempA, index=tempL1, columns=tempL2)
tempDF['Sex'] = protDF_B['Sex']

#Update
protDF_B = tempDF
display(protDF_B.describe(include='all'))

### 3-2. Correlation b/w all pairwise variables

In [ ]:
#Female and male cohorts
tempL = [item for sublist in [covarL, ['log_BaseProtBMI']] for item in sublist]

print('Female cohort:')
#Compute correlation matrix
tempDF1 = protDF_F.drop(columns=tempL)
tempDF1 = tempDF1.corr(method='pearson')
print('• Combinations:', int(len(tempDF1)*(len(tempDF1)-1)/2))

#Extract lower triangle matrix
tempDF1 = tempDF1.where(np.tril(np.ones(tempDF1.shape), k=-1).astype(np.bool), other=np.nan)
tempDF1.index.set_names('Variable1', inplace=True)
tempDF1 = tempDF1.reset_index().melt(var_name='Variable2', value_name='Pearson_r', id_vars=['Variable1'])
tempDF1 = tempDF1.dropna()
print('• nrows:', len(tempDF1))
print('• |Pearson\'s r| > 0.8:', len(tempDF1.loc[abs(tempDF1['Pearson_r'])>0.8]))
display(tempDF1.loc[abs(tempDF1['Pearson_r'])>0.8])

print('Male cohort:')
#Compute correlation matrix
tempDF2 = protDF_M.drop(columns=tempL)
tempDF2 = tempDF2.corr(method='pearson')
print('• Combinations:', int(len(tempDF2)*(len(tempDF2)-1)/2))

#Extract lower triangle matrix
tempDF2 = tempDF2.where(np.tril(np.ones(tempDF2.shape), k=-1).astype(np.bool), other=np.nan)
tempDF2.index.set_names('Variable1', inplace=True)
tempDF2 = tempDF2.reset_index().melt(var_name='Variable2', value_name='Pearson_r', id_vars=['Variable1'])
tempDF2 = tempDF2.dropna()
print('• nrows:', len(tempDF2))
print('• |Pearson\'s r| > 0.8:', len(tempDF2.loc[abs(tempDF2['Pearson_r'])>0.8]))
display(tempDF2.loc[abs(tempDF2['Pearson_r'])>0.8])

#Distribution
sns.set(style='ticks', font='Arial', context='notebook')
plt.figure(figsize=(4, 3))
sns.distplot(tempDF1['Pearson_r'], color='r', label='Female').set(xlim=(-1, 1), xticks=np.arange(-1, 1.1, 0.5))
sns.distplot(tempDF2['Pearson_r'], color='b', label='Male').set(xlim=(-1, 1), xticks=np.arange(-1, 1.1, 0.5))
sns.despine()
plt.xlabel('Pearson\'s '+r'$r$')
plt.ylabel('Density')
plt.legend(bbox_to_anchor=(1, 0.5), loc='center left', borderaxespad=1)
plt.show()

In [ ]:
#Both sex cohort
tempL = [item for sublist in [covarL, ['log_BaseProtBMI']] for item in sublist]

print('Both sex cohort:')
#Compute correlation matrix
tempDF = protDF_B.drop(columns=tempL)
tempDF = tempDF.corr(method='pearson')
print('• Combinations:', int(len(tempDF)*(len(tempDF)-1)/2))

#Extract lower triangle matrix
tempDF = tempDF.where(np.tril(np.ones(tempDF.shape), k=-1).astype(np.bool), other=np.nan)
tempDF.index.set_names('Variable1', inplace=True)
tempDF = tempDF.reset_index().melt(var_name='Variable2', value_name='Pearson_r', id_vars=['Variable1'])
tempDF = tempDF.dropna()
print('• nrows:', len(tempDF))
print('• |Pearson\'s r| > 0.8:', len(tempDF.loc[abs(tempDF['Pearson_r'])>0.8]))
display(tempDF.loc[abs(tempDF['Pearson_r'])>0.8])

#Distribution
sns.set(style='ticks', font='Arial', context='talk')
plt.figure(figsize=(4, 3))
sns.distplot(tempDF['Pearson_r'], color='r').set(xlim=(-1, 1), xticks=np.arange(-1, 1.1, 0.5))
sns.despine()
plt.ylabel('Density')
plt.xlabel('Pearson\'s '+r'$r$')
plt.axvline(x=tempDF['Pearson_r'].max(), **{'linestyle':'--', 'color':'r'})
plt.axvline(x=tempDF['Pearson_r'].min(), **{'linestyle':'--', 'color':'r'})
plt.show()

### 3-3. Beta-coefficients of LASSO models

In [ ]:
#Female model
#Import the cleaned beta-coefficients of LASSO
fileDir = './ExportData/'
fileName = '210105_Biological-BMI-paper_baseline-LASSO_protBMI-Female_LASSO-bcoefs.tsv'
protBMI_F_bcoefs = pd.read_csv(fileDir+fileName, sep='\t').set_index('Variable')
print('Variables:', len(protBMI_F_bcoefs))

#Variables with non-zero beta-coefficient
tempDF = protBMI_F_bcoefs.loc[protBMI_F_bcoefs['nZeros']!=10]
print('Variables with non-zero beta-coefficient:', len(tempDF),
      '(', len(tempDF)/len(protBMI_F_bcoefs)*100, '%)')

#Variables with non-zero beta-coefficient in all 10 models
tempDF = protBMI_F_bcoefs.loc[protBMI_F_bcoefs['nZeros']==0]
tempDF = tempDF.sort_values(by='MEAN', ascending=False)
print('Variables with non-zero beta-coefficient in all 10 models:', len(tempDF),
      '(', len(tempDF)/len(protBMI_F_bcoefs)*100, '%)')
display(tempDF)

In [ ]:
#Male model
#Import the cleaned beta-coefficients of LASSO
fileDir = './ExportData/'
fileName = '210105_Biological-BMI-paper_baseline-LASSO_protBMI-Male_LASSO-bcoefs.tsv'
protBMI_M_bcoefs = pd.read_csv(fileDir+fileName, sep='\t').set_index('Variable')
print('Variables:', len(protBMI_M_bcoefs))

#Variables with non-zero beta-coefficient
tempDF = protBMI_M_bcoefs.loc[protBMI_M_bcoefs['nZeros']!=10]
print('Variables with non-zero beta-coefficient:', len(tempDF),
      '(', len(tempDF)/len(protBMI_M_bcoefs)*100, '%)')

#Variables with non-zero beta-coefficient in all 10 models
tempDF = protBMI_M_bcoefs.loc[protBMI_M_bcoefs['nZeros']==0]
tempDF = tempDF.sort_values(by='MEAN', ascending=False)
print('Variables with non-zero beta-coefficient in all 10 models:', len(tempDF),
      '(', len(tempDF)/len(protBMI_M_bcoefs)*100, '%)')
display(tempDF)

In [ ]:
#Both sex model
#Import the cleaned beta-coefficients of LASSO
fileDir = './ExportData/'
fileName = '210105_Biological-BMI-paper_baseline-LASSO_protBMI-BothSex_LASSO-bcoefs.tsv'
protBMI_B_bcoefs = pd.read_csv(fileDir+fileName, sep='\t').set_index('Variable')
print('Variables:', len(protBMI_B_bcoefs))

#Variables with non-zero beta-coefficient
tempDF = protBMI_B_bcoefs.loc[protBMI_B_bcoefs['nZeros']!=10]
print('Variables with non-zero beta-coefficient:', len(tempDF),
      '(', len(tempDF)/len(protBMI_B_bcoefs)*100, '%)')

#Variables with non-zero beta-coefficient in all 10 models
tempDF = protBMI_B_bcoefs.loc[protBMI_B_bcoefs['nZeros']==0]
tempDF = tempDF.sort_values(by='MEAN', ascending=False)
print('Variables with non-zero beta-coefficient in all 10 models:', len(tempDF),
      '(', len(tempDF)/len(protBMI_B_bcoefs)*100, '%)')
display(tempDF)

### 3-4. Variables with non-zero beta-coefficient in all 10 models

In [ ]:
#Variables with non-zero beta-coefficient in all 10 models
tempS1 = set(protBMI_F_bcoefs.loc[protBMI_F_bcoefs['nZeros']==0].index)
tempS2 = set(protBMI_M_bcoefs.loc[protBMI_M_bcoefs['nZeros']==0].index)
tempS3 = set(protBMI_B_bcoefs.loc[protBMI_B_bcoefs['nZeros']==0].index)

#Common the robust variables
var_111 = tempS1 & tempS2 & tempS3
print('Common variables with non-zero beta-coefficient in all 10 models:\n', var_111)
var_110 = (tempS1 & tempS2) - var_111
var_101 = (tempS1 & tempS3) - var_111
var_011 = (tempS2 & tempS3) - var_111
var_100 = tempS1 - (var_110 | var_101 | var_111)
var_010 = tempS2 - (var_110 | var_011 | var_111)
var_001 = tempS3 - (var_101 | var_011 | var_111)

#The number of each subset
var_subset = [len(var_100), len(var_010), len(var_110), len(var_001),
              len(var_101), len(var_011), len(var_111)]
print('Each subset:', var_subset)

#Venn diagram
sns.set(font='Arial', context='talk')
venn3(subsets=var_subset, set_labels=('Female', 'Male', 'Both sex'), set_colors=('r', 'b', 'g'), alpha=0.4)
venn3_circles(subsets=var_subset)
plt.title('Robust variables\n—Proteomics—', fontdict={'fontsize':24})
plt.show()

In [ ]:
#Variables with non-zero beta-coefficient in all 10 models
tempS1 = set(protBMI_F_bcoefs.loc[protBMI_F_bcoefs['nZeros']==0].index)
tempS2 = set(protBMI_M_bcoefs.loc[protBMI_M_bcoefs['nZeros']==0].index)
tempS3 = set(protBMI_B_bcoefs.loc[protBMI_B_bcoefs['nZeros']==0].index)

#Union of the robust variables
tempL = list(tempS1 | tempS2 | tempS3)
print('Union of the variables with non-zero beta-coefficient in all 10 models:', len(tempL))

#Merge (use original bcoefs to rescue ones with zeros in any 10 models)
tempDF1 = protBMI_F_bcoefs.loc[protBMI_F_bcoefs.index.isin(tempL)].drop(columns=['MEAN', 'SD', 'nZeros'])
tempDF2 = protBMI_M_bcoefs.loc[protBMI_M_bcoefs.index.isin(tempL)].drop(columns=['MEAN', 'SD', 'nZeros'])
tempDF3 = protBMI_B_bcoefs.loc[protBMI_B_bcoefs.index.isin(tempL)].drop(columns=['MEAN', 'SD', 'nZeros'])
tempDF = pd.concat([tempDF1, tempDF2, tempDF3], axis=0)
tempDF['Model'] = np.repeat(['Female', 'Male', 'Both sex'], [len(tempDF1), len(tempDF2), len(tempDF3)])

#tidyr::gather in R
tempDF1 = tempDF.reset_index().melt(var_name='LASSO', value_name='bcoef', id_vars=['Variable', 'Model'])

#Order for plot
tempDF2 = tempDF1.loc[tempDF1['Model']=='Both sex'].groupby('Variable', as_index=True).agg({'bcoef': np.mean})
tempDF2 = tempDF2.sort_values(by='bcoef', ascending=False)

#Plot
sns.set(style='ticks', font='Arial', context='talk')
sns.catplot(data=tempDF1, y='Variable', order=tempDF2.index.tolist(), x='bcoef', kind='box',
                hue='Model', palette='Set1', dodge=True, height=14, aspect=0.6,
                showfliers=True, flierprops={'marker':'o', 'markerfacecolor':'gray', 'alpha':0.4})
plt.xlabel(r'$\beta$'+'-coefficient in LASSO models')
plt.ylabel('')
tempA = np.arange(-0.04, 0.081, 0.02)
for coord in tempA:
    plt.axvline(x=coord, **{'linestyle':'--', 'color':'gray'})
for row_i in range(len(tempDF2)):
    if row_i%2 == 0:
        plt.axhspan(ymin=row_i-0.5, ymax=row_i+0.5, facecolor='r', alpha=0.1)
plt.show()

In [ ]:
#Both sex model

#Variables with non-zero beta-coefficient in all 10 models
tempDF = protBMI_B_bcoefs.loc[protBMI_B_bcoefs['nZeros']==0].sort_values(by='MEAN', ascending=False)
tempDF = tempDF.drop(columns=['MEAN', 'SD', 'nZeros'])
print('Variables with non-zero beta-coefficient in all 10 models:', len(tempDF))

#tidyr::gather in R
tempDF1 = tempDF.reset_index().melt(var_name='LASSO', value_name='bcoef', id_vars=['Variable'])

#Plot
sns.set(style='ticks', font='Arial', context='talk')
plt.figure(figsize=(5, 10))
p = sns.boxplot(data=tempDF1, y='Variable', x='bcoef', color='r', dodge=False,
                showfliers=True, flierprops={'marker':'o', 'markerfacecolor':'gray', 'alpha':0.4},
                showcaps=True, notch=False)
p.set(xlim=(-0.046, 0.056), xticks=np.arange(-0.04, 0.06, 0.02))
p.grid(axis='x', linestyle='--', color='black')
sns.despine()
plt.ylabel('')
plt.xlabel(r'$\beta$'+'-coefficient in LASSO model')
for row_i in range(len(tempDF)):
    if row_i%2 == 0:
        plt.axhspan(ymin=row_i-0.5, ymax=row_i+0.5, facecolor='r', alpha=0.2, zorder=0)
##Save
fileDir = './ExportFigures/'
fileName = '210823_Biological-BMI-paper_LASSO-bcoef_protBMI-BothSex-bcoef_non-zero-in-all.tif'
plt.gcf().savefig(fileDir+fileName, dpi=300, bbox_inches='tight', pad_inches=0,
                  pil_kwargs={'compression':'tiff_lzw'})
plt.show()

### 3-5. Correlation b/w pairwise variables with non-zero beta coefficient in all 10 models

In [ ]:
#Both sex model
print('Both sex model:')

#Variables with non-zero beta-coefficient in all 10 models
tempL = protBMI_B_bcoefs.loc[protBMI_B_bcoefs['nZeros']==0].index.tolist()

#Compute correlation matrix
tempDF = protDF_B[tempL]
tempDF = tempDF.corr(method='pearson')
print('• Combinations:', int(len(tempDF)*(len(tempDF)-1)/2))

#Extract lower triangle matrix
tempDF = tempDF.where(np.tril(np.ones(tempDF.shape), k=-1).astype(np.bool), other=np.nan)
tempDF.index.set_names('Variable1', inplace=True)
tempDF = tempDF.reset_index().melt(var_name='Variable2', value_name='Pearson_r', id_vars=['Variable1'])
tempDF = tempDF.dropna()
print('• nrows:', len(tempDF))
print('• |Pearson\'s r| > 0.8:', len(tempDF.loc[abs(tempDF['Pearson_r'])>0.8]))
display(tempDF.loc[abs(tempDF['Pearson_r'])>0.8])

#Distribution
sns.set(style='ticks', font='Arial', context='talk')
plt.figure(figsize=(4, 3))
sns.distplot(tempDF['Pearson_r'], color='r').set(xlim=(-1, 1), xticks=np.arange(-1, 1.1, 0.5))
sns.despine()
plt.ylabel('Density')
plt.xlabel('Pearson\'s '+r'$r$')
plt.axvline(x=tempDF['Pearson_r'].max(), **{'linestyle':'--', 'color':'r'})
plt.axvline(x=tempDF['Pearson_r'].min(), **{'linestyle':'--', 'color':'r'})
plt.show()

print('Variables with non-zero beta-coefficient in all 10 models:', len(tempL))
#Clustermap
tempDF = protDF_B[tempL]
tempDF = tempDF.corr(method='pearson')
sns.set(style='ticks', font='Arial', context='notebook')
cmap = sns.diverging_palette(220, 20, as_cmap=True)
cm = sns.clustermap(tempDF, method='ward', metric='euclidean', cmap=cmap,
                    row_cluster=True, col_cluster=True, row_linkage=None, col_linkage=None,
                    row_colors=None, col_colors=None, xticklabels=True, yticklabels=True,
                    dendrogram_ratio=(0.15, 0.15), cbar_pos=(0.92, 0.02, 0.02, 0.06),
                    figsize=(7, 7), **{'center':0, 'vmin':-1, 'vmax':1})
cm.cax.set_title('Pearson\'s '+r'$r$')
hm = cm.ax_heatmap.get_position()
rd = cm.ax_row_dendrogram.get_position()
cd = cm.ax_col_dendrogram.get_position()
cm.ax_heatmap.set_position([hm.x0, hm.y0, hm.width, hm.height])
cm.ax_row_dendrogram.set_position([rd.x0+rd.width*0.5, rd.y0, rd.width*0.5, rd.height])
cm.ax_col_dendrogram.set_position([cd.x0, cd.y0, cd.width, cd.height*0.5])
plt.show()

### 3-6. Explained variance of variables with non-zero beta-coefficient

> OLS linear regression model for significance:  
> ***BMI ~ b0 + b1\*Variable + b2\*Sex + b3\*BaseAge + b4\*PCs***  
> Univariate model:  
> ***BMI ~ b0 + b1\*Variable***  
> –> Biological BMI is included for comparison.  

In [ ]:
#Female model
print('Female model:')

#Variables with non-zero beta-coefficient at least 1 of 10 models
tempL = protBMI_F_bcoefs.loc[protBMI_F_bcoefs['nZeros']!=10].index.tolist()

#Prepare dataframe for OLS regressions
tempL = [item for sublist in [tempL, covarL, ['log_BaseProtBMI']] for item in sublist]
tempDF = protDF_F[tempL]
tempDF = tempDF.rename(columns={'log_BaseProtBMI': 'ProtBMI model'})

#Extract independent variables in OLS regressions (including protBMI)
tempL = tempDF.drop(columns=covarL).columns.tolist()

#Perform OLS regressions
tempL1 = []#For R2
tempL2 = []#For beta-coefficient
tempL3 = []#For SE of beta-coefficient
tempL4 = []#For p-value
tempL5 = []#For R2 in univariate model
t_start = time.time()
for var in tempL:
    tempDF_ols = tempDF[[item for sublist in [covarL, [var]] for item in sublist]]
    tempDF_ols = tempDF_ols.rename(columns={var: 'Variable'})
    #Add a constant for the intercept -> Similar to R, smf automatically add a constant

    #Univariate model
    #Fit model
    formula = 'log_BaseBMI ~ Variable'
    resOLS = smf.ols(formula, data=tempDF_ols).fit()
    #Save R2 [%]
    tempL5.append(resOLS.rsquared*100)
    
    #Full model
    #Fit model (Sex is invariant in this model and eliminated)
    formula = 'log_BaseBMI ~ Variable + BaseAge + PC1 + PC2 + PC3 + PC4 + PC5'
    resOLS = smf.ols(formula, data=tempDF_ols).fit()
    #Save R2 [%]
    tempL1.append(resOLS.rsquared*100)
    #Save beta-coefficient of the variable
    tempL2.append(resOLS.params['Variable'])
    tempL3.append(resOLS.bse['Variable'])
    #Save p-value of the variable
    tempL4.append(resOLS.pvalues['Variable'])
t_elapsed = time.time() - t_start
print('Elapsed time for ', len(tempL), 'OLS regressions (including protBMI):',
      round(t_elapsed//60), 'min', round(t_elapsed%60, 1), 'sec')

#Clean the results
tempDF = pd.DataFrame({'UnivarR2':tempL5, 'R2':tempL1, 'OLSbcoef':tempL2, 'OLSbcoefSE':tempL3, 'Pval':tempL4},
                      index=tempL)
tempDF.index.set_names('Variable', inplace=True)
##P-value adjustment by using Benjamini–Hochberg method
tempDF['AdjPval'] = multi.multipletests(tempDF['Pval'], alpha=0.05, method='fdr_bh',
                                        is_sorted=False, returnsorted=False)[1]
##Add the LASSO results
tempDF['LASSObcoef'] = protBMI_F_bcoefs.loc[protBMI_F_bcoefs.index.isin(tempL)]['MEAN']#NaN in protBMI reference
tempDF['LASSOnZeros'] = protBMI_F_bcoefs.loc[protBMI_F_bcoefs.index.isin(tempL)]['nZeros']#NaN in protBMI reference
tempDF = tempDF.sort_values(by='AdjPval', ascending=True)

#Save
fileDir = './ExportData/'
fileName = '210105_Biological-BMI-paper_LASSO-bcoef_OLS_protBMI-Female.tsv'
tempDF.to_csv(fileDir+fileName, sep='\t', index=True)

#Extact significant variables
tempDF = tempDF[tempDF['AdjPval']<0.05]
print('Variables significantly associated with BMI (FDR < 0.05):', len(tempDF)-1)#protBMI reference

#Top 30 (+ protBMI)
topX = 30+1
display(tempDF.iloc[0:topX])
##Category
tempL = []
for row_n in tempDF.index.tolist():
    if row_n=='ProtBMI model':
        tempL.append('Positive association')
    else:
        if tempDF.loc[row_n, 'OLSbcoef'] > 0:
            tempL.append('Positive association')
        elif tempDF.loc[row_n, 'OLSbcoef'] < 0:
            tempL.append('Negative association')
        else:
            tempL.append('Error')
tempDF['Association'] = tempL
##Plot
tempDF = tempDF.reset_index()
sns.set(style='ticks', font='Arial', context='talk')
plt.figure(figsize=(10, 4))
p = sns.barplot(data=tempDF.iloc[0:topX], y='UnivarR2', x='Variable', hue='Association', dodge=False,
                palette={'Positive association':'tab:red', 'Negative association':'tab:blue'}, edgecolor='k')
p.set(ylim=(0, 80), yticks=np.arange(0, 80, 15))
p.grid(axis='y', linestyle='--', color='black')
sns.despine()
plt.xticks(rotation=90, horizontalalignment='center')
plt.xlabel('')
plt.ylabel('Ratio of explained variance [%]')
plt.legend(loc='upper right')
plt.show()

In [ ]:
#Male model
print('Male model:')

#Variables with non-zero beta-coefficient at least 1 of 10 models
tempL = protBMI_M_bcoefs.loc[protBMI_M_bcoefs['nZeros']!=10].index.tolist()

#Prepare dataframe for OLS regressions
tempL = [item for sublist in [tempL, covarL, ['log_BaseProtBMI']] for item in sublist]
tempDF = protDF_M[tempL]
tempDF = tempDF.rename(columns={'log_BaseProtBMI': 'ProtBMI model'})

#Extract independent variables in OLS regressions (including protBMI)
tempL = tempDF.drop(columns=covarL).columns.tolist()

#Perform OLS regressions
tempL1 = []#For R2
tempL2 = []#For beta-coefficient
tempL3 = []#For SE of beta-coefficient
tempL4 = []#For p-value
tempL5 = []#For R2 in univariate model
t_start = time.time()
for var in tempL:
    tempDF_ols = tempDF[[item for sublist in [covarL, [var]] for item in sublist]]
    tempDF_ols = tempDF_ols.rename(columns={var: 'Variable'})
    #Add a constant for the intercept -> Similar to R, smf automatically add a constant

    #Univariate model
    #Fit model
    formula = 'log_BaseBMI ~ Variable'
    resOLS = smf.ols(formula, data=tempDF_ols).fit()
    #Save R2 [%]
    tempL5.append(resOLS.rsquared*100)
    
    #Full model
    #Fit model (Sex is invariant in this model and eliminated)
    formula = 'log_BaseBMI ~ Variable + BaseAge + PC1 + PC2 + PC3 + PC4 + PC5'
    resOLS = smf.ols(formula, data=tempDF_ols).fit()
    #Save R2 [%]
    tempL1.append(resOLS.rsquared*100)
    #Save beta-coefficient of the variable
    tempL2.append(resOLS.params['Variable'])
    tempL3.append(resOLS.bse['Variable'])
    #Save p-value of the variable
    tempL4.append(resOLS.pvalues['Variable'])
t_elapsed = time.time() - t_start
print('Elapsed time for ', len(tempL), 'OLS regressions (including protBMI):',
      round(t_elapsed//60), 'min', round(t_elapsed%60, 1), 'sec')

#Clean the results
tempDF = pd.DataFrame({'UnivarR2':tempL5, 'R2':tempL1, 'OLSbcoef':tempL2, 'OLSbcoefSE':tempL3, 'Pval':tempL4},
                      index=tempL)
tempDF.index.set_names('Variable', inplace=True)
##P-value adjustment by using Benjamini–Hochberg method
tempDF['AdjPval'] = multi.multipletests(tempDF['Pval'], alpha=0.05, method='fdr_bh',
                                        is_sorted=False, returnsorted=False)[1]
##Add the LASSO results
tempDF['LASSObcoef'] = protBMI_M_bcoefs.loc[protBMI_M_bcoefs.index.isin(tempL)]['MEAN']#NaN in protBMI reference
tempDF['LASSOnZeros'] = protBMI_M_bcoefs.loc[protBMI_M_bcoefs.index.isin(tempL)]['nZeros']#NaN in protBMI reference
tempDF = tempDF.sort_values(by='AdjPval', ascending=True)

#Save
fileDir = './ExportData/'
fileName = '210105_Biological-BMI-paper_LASSO-bcoef_OLS_protBMI-Male.tsv'
tempDF.to_csv(fileDir+fileName, sep='\t', index=True)

#Extact significant variables
tempDF = tempDF[tempDF['AdjPval']<0.05]
print('Variables significantly associated with BMI (FDR < 0.05):', len(tempDF)-1)#protBMI reference

#Top 30 (+ protBMI)
topX = 30+1
display(tempDF.iloc[0:topX])
##Category
tempL = []
for row_n in tempDF.index.tolist():
    if row_n=='ProtBMI model':
        tempL.append('Positive association')
    else:
        if tempDF.loc[row_n, 'OLSbcoef'] > 0:
            tempL.append('Positive association')
        elif tempDF.loc[row_n, 'OLSbcoef'] < 0:
            tempL.append('Negative association')
        else:
            tempL.append('Error')
tempDF['Association'] = tempL
##Plot
tempDF = tempDF.reset_index()
sns.set(style='ticks', font='Arial', context='talk')
plt.figure(figsize=(10, 4))
p = sns.barplot(data=tempDF.iloc[0:topX], y='UnivarR2', x='Variable', hue='Association', dodge=False,
                palette={'Positive association':'tab:red', 'Negative association':'tab:blue'}, edgecolor='k')
p.set(ylim=(0, 80), yticks=np.arange(0, 80, 15))
p.grid(axis='y', linestyle='--', color='black')
sns.despine()
plt.xticks(rotation=90, horizontalalignment='center')
plt.xlabel('')
plt.ylabel('Ratio of explained variance [%]')
plt.legend(loc='upper right')
plt.show()

In [ ]:
#Both sex model
print('Both sex model:')

#Variables with non-zero beta-coefficient at least 1 of 10 models
tempL = protBMI_B_bcoefs.loc[protBMI_B_bcoefs['nZeros']!=10].index.tolist()

#Prepare dataframe for OLS regressions
tempL = [item for sublist in [tempL, covarL, ['log_BaseProtBMI']] for item in sublist]
tempDF = protDF_B[tempL]
tempDF = tempDF.rename(columns={'log_BaseProtBMI': 'ProtBMI model'})

#Extract independent variables in OLS regressions (including protBMI)
tempL = tempDF.drop(columns=covarL).columns.tolist()

#Perform OLS regressions
tempL1 = []#For R2
tempL2 = []#For beta-coefficient
tempL3 = []#For SE of beta-coefficient
tempL4 = []#For p-value
tempL5 = []#For R2 in univariate model
t_start = time.time()
for var in tempL:
    tempDF_ols = tempDF[[item for sublist in [covarL, [var]] for item in sublist]]
    tempDF_ols = tempDF_ols.rename(columns={var: 'Variable'})
    #Add a constant for the intercept -> Similar to R, smf automatically add a constant

    #Univariate model
    #Fit model
    formula = 'log_BaseBMI ~ Variable'
    resOLS = smf.ols(formula, data=tempDF_ols).fit()
    #Save R2 [%]
    tempL5.append(resOLS.rsquared*100)
    
    #Full model
    #Fit model
    formula = 'log_BaseBMI ~ Variable + C(Sex) + BaseAge + PC1 + PC2 + PC3 + PC4 + PC5'
    resOLS = smf.ols(formula, data=tempDF_ols).fit()
    #Save R2 [%]
    tempL1.append(resOLS.rsquared*100)
    #Save beta-coefficient of the variable
    tempL2.append(resOLS.params['Variable'])
    tempL3.append(resOLS.bse['Variable'])
    #Save p-value of the variable
    tempL4.append(resOLS.pvalues['Variable'])
t_elapsed = time.time() - t_start
print('Elapsed time for ', len(tempL), 'OLS regressions (including protBMI):',
      round(t_elapsed//60), 'min', round(t_elapsed%60, 1), 'sec')

#Clean the results
tempDF = pd.DataFrame({'UnivarR2':tempL5, 'R2':tempL1, 'OLSbcoef':tempL2, 'OLSbcoefSE':tempL3, 'Pval':tempL4},
                      index=tempL)
tempDF.index.set_names('Variable', inplace=True)
##P-value adjustment by using Benjamini–Hochberg method
tempDF['AdjPval'] = multi.multipletests(tempDF['Pval'], alpha=0.05, method='fdr_bh',
                                        is_sorted=False, returnsorted=False)[1]
##Add the LASSO results
tempDF['LASSObcoef'] = protBMI_B_bcoefs.loc[protBMI_B_bcoefs.index.isin(tempL)]['MEAN']#NaN in protBMI reference
tempDF['LASSOnZeros'] = protBMI_B_bcoefs.loc[protBMI_B_bcoefs.index.isin(tempL)]['nZeros']#NaN in protBMI reference
tempDF = tempDF.sort_values(by='AdjPval', ascending=True)

#Save
fileDir = './ExportData/'
fileName = '210105_Biological-BMI-paper_LASSO-bcoef_OLS_protBMI-BothSex.tsv'
tempDF.to_csv(fileDir+fileName, sep='\t', index=True)

#Extact significant variables
tempDF = tempDF[tempDF['AdjPval']<0.05]
print('Variables significantly associated with BMI (FDR < 0.05):', len(tempDF)-1)#protBMI reference

#Top 30 (+ protBMI)
topX = 30+1
display(tempDF.iloc[0:topX])
##Category
tempL = []
for row_n in tempDF.index.tolist():
    if row_n=='ProtBMI model':
        tempL.append('Positive association')
    else:
        if tempDF.loc[row_n, 'OLSbcoef'] > 0:
            tempL.append('Positive association')
        elif tempDF.loc[row_n, 'OLSbcoef'] < 0:
            tempL.append('Negative association')
        else:
            tempL.append('Error')
tempDF['Association'] = tempL
##Plot
tempDF = tempDF.reset_index()
sns.set(style='ticks', font='Arial', context='talk')
plt.figure(figsize=(10, 4))
p = sns.barplot(data=tempDF.iloc[0:topX], y='UnivarR2', x='Variable', hue='Association', dodge=False,
                palette={'Positive association':'tab:red', 'Negative association':'tab:blue'}, edgecolor='k')
p.set(ylim=(0, 80), yticks=np.arange(0, 80, 15))
p.grid(axis='y', linestyle='--', color='black')
sns.despine()
plt.xticks(rotation=90, horizontalalignment='center')
plt.xlabel('')
plt.ylabel('Explained variance in BMI [%]')
plt.legend(loc='upper right')
##Save
fileDir = './ExportFigures/'
fileName = '210823_Biological-BMI-paper_LASSO-bcoef_OLS_protBMI-BothSex_top30.tif'
plt.gcf().savefig(fileDir+fileName, dpi=300, bbox_inches='tight', pad_inches=0,
                  pil_kwargs={'compression':'tiff_lzw'})
plt.show()

## 4. Metabolomics, Clinical labs, Proteomics-combined omics

### 4-1. Prepare variables

> Variables are standardized for OLS.

In [ ]:
#Import the cleaned baseline dataframe for LASSO
fileDir = './ExportData/'
fileName = '210104_Biological-BMI-paper_RF-imputation_baseline-combiDF-with-RF-imputation.tsv'
tempDF = pd.read_csv(fileDir+fileName, sep='\t', dtype={'public_client_id': str}).set_index('public_client_id')

#Splitting dataset into sex-dependent cohorts
combiDF_F = tempDF.loc[tempDF['Sex']=='F']
combiDF_M = tempDF.loc[tempDF['Sex']=='M']
combiDF_B = tempDF#Not copy just rename
print('(Female, Male, Both): (', len(combiDF_F.index), ', ', len(combiDF_M.index), ', ', len(combiDF_B.index), ')')

In [ ]:
#Female models

#Import biological BMI
fileDir = './ExportData/'
fileName = '210105_Biological-BMI-paper_baseline-LASSO_combiBMI-Female.csv'
tempDF = pd.read_csv(fileDir+fileName, dtype={'public_client_id': str}).set_index('public_client_id')
combiDF_F['log_BaseCombiBMI'] = tempDF['log_BaseCombiBMI']

#Z-score standardization
tempDF = combiDF_F.drop(columns=['Sex', 'Race'])
tempL1 = tempDF.index.tolist()
tempL2 = tempDF.columns.tolist()
scaler = StandardScaler(copy=True, with_mean=True, with_std=True)
tempA = scaler.fit_transform(tempDF)#Column direction
tempDF = pd.DataFrame(data=tempA, index=tempL1, columns=tempL2)
tempDF['Sex'] = combiDF_F['Sex']

#Update
combiDF_F = tempDF
display(combiDF_F.describe(include='all'))

In [ ]:
#Male model

#Import biological BMI
fileDir = './ExportData/'
fileName = '210105_Biological-BMI-paper_baseline-LASSO_combiBMI-Male.csv'
tempDF = pd.read_csv(fileDir+fileName, dtype={'public_client_id': str}).set_index('public_client_id')
combiDF_M['log_BaseCombiBMI'] = tempDF['log_BaseCombiBMI']

#Z-score standardization
tempDF = combiDF_M.drop(columns=['Sex', 'Race'])
tempL1 = tempDF.index.tolist()
tempL2 = tempDF.columns.tolist()
scaler = StandardScaler(copy=True, with_mean=True, with_std=True)
tempA = scaler.fit_transform(tempDF)#Column direction
tempDF = pd.DataFrame(data=tempA, index=tempL1, columns=tempL2)
tempDF['Sex'] = combiDF_M['Sex']

#Update
combiDF_M = tempDF
display(combiDF_M.describe(include='all'))

In [ ]:
#Both sex model

#Import biological BMI
fileDir = './ExportData/'
fileName = '210105_Biological-BMI-paper_baseline-LASSO_combiBMI-BothSex.csv'
tempDF = pd.read_csv(fileDir+fileName, dtype={'public_client_id': str}).set_index('public_client_id')
combiDF_B['log_BaseCombiBMI'] = tempDF['log_BaseCombiBMI']

#Z-score standardization
tempDF = combiDF_B.drop(columns=['Sex', 'Race'])
tempL1 = tempDF.index.tolist()
tempL2 = tempDF.columns.tolist()
scaler = StandardScaler(copy=True, with_mean=True, with_std=True)
tempA = scaler.fit_transform(tempDF)#Column direction
tempDF = pd.DataFrame(data=tempA, index=tempL1, columns=tempL2)
tempDF['Sex'] = combiDF_B['Sex']

#Update
combiDF_B = tempDF
display(combiDF_B.describe(include='all'))

### 4-2. Correlation b/w all pairwise variables

In [ ]:
#Female and male cohorts
tempL = [item for sublist in [covarL, ['log_BaseCombiBMI']] for item in sublist]

print('Female cohort:')
#Compute correlation matrix
tempDF1 = combiDF_F.drop(columns=tempL)
tempDF1 = tempDF1.corr(method='pearson')
print('• Combinations:', int(len(tempDF1)*(len(tempDF1)-1)/2))

#Extract lower triangle matrix
tempDF1 = tempDF1.where(np.tril(np.ones(tempDF1.shape), k=-1).astype(np.bool), other=np.nan)
tempDF1.index.set_names('Variable1', inplace=True)
tempDF1 = tempDF1.reset_index().melt(var_name='Variable2', value_name='Pearson_r', id_vars=['Variable1'])
tempDF1 = tempDF1.dropna()
print('• nrows:', len(tempDF1))
print('• |Pearson\'s r| > 0.8:', len(tempDF1.loc[abs(tempDF1['Pearson_r'])>0.8]))
display(tempDF1.loc[abs(tempDF1['Pearson_r'])>0.8])

print('Male cohort:')
#Compute correlation matrix
tempDF2 = combiDF_M.drop(columns=tempL)
tempDF2 = tempDF2.corr(method='pearson')
print('• Combinations:', int(len(tempDF2)*(len(tempDF2)-1)/2))

#Extract lower triangle matrix
tempDF2 = tempDF2.where(np.tril(np.ones(tempDF2.shape), k=-1).astype(np.bool), other=np.nan)
tempDF2.index.set_names('Variable1', inplace=True)
tempDF2 = tempDF2.reset_index().melt(var_name='Variable2', value_name='Pearson_r', id_vars=['Variable1'])
tempDF2 = tempDF2.dropna()
print('• nrows:', len(tempDF2))
print('• |Pearson\'s r| > 0.8:', len(tempDF2.loc[abs(tempDF2['Pearson_r'])>0.8]))
display(tempDF2.loc[abs(tempDF2['Pearson_r'])>0.8])

#Distribution
sns.set(style='ticks', font='Arial', context='notebook')
plt.figure(figsize=(4, 3))
sns.distplot(tempDF1['Pearson_r'], color='r', label='Female').set(xlim=(-1, 1), xticks=np.arange(-1, 1.1, 0.5))
sns.distplot(tempDF2['Pearson_r'], color='b', label='Male').set(xlim=(-1, 1), xticks=np.arange(-1, 1.1, 0.5))
sns.despine()
plt.xlabel('Pearson\'s '+r'$r$')
plt.ylabel('Density')
plt.legend(bbox_to_anchor=(1, 0.5), loc='center left', borderaxespad=1)
plt.show()

In [ ]:
#Both sex cohort
tempL = [item for sublist in [covarL, ['log_BaseCombiBMI']] for item in sublist]

print('Both sex cohort:')
#Compute correlation matrix
tempDF = combiDF_B.drop(columns=tempL)
tempDF = tempDF.corr(method='pearson')
print('• Combinations:', int(len(tempDF)*(len(tempDF)-1)/2))

#Extract lower triangle matrix
tempDF = tempDF.where(np.tril(np.ones(tempDF.shape), k=-1).astype(np.bool), other=np.nan)
tempDF.index.set_names('Variable1', inplace=True)
tempDF = tempDF.reset_index().melt(var_name='Variable2', value_name='Pearson_r', id_vars=['Variable1'])
tempDF = tempDF.dropna()
print('• nrows:', len(tempDF))
print('• |Pearson\'s r| > 0.8:', len(tempDF.loc[abs(tempDF['Pearson_r'])>0.8]))
display(tempDF.loc[abs(tempDF['Pearson_r'])>0.8])

#Distribution
sns.set(style='ticks', font='Arial', context='talk')
plt.figure(figsize=(4, 3))
sns.distplot(tempDF['Pearson_r'], color='m').set(xlim=(-1, 1), xticks=np.arange(-1, 1.1, 0.5))
sns.despine()
plt.ylabel('Density')
plt.xlabel('Pearson\'s '+r'$r$')
plt.axvline(x=tempDF['Pearson_r'].max(), **{'linestyle':'--', 'color':'m'})
plt.axvline(x=tempDF['Pearson_r'].min(), **{'linestyle':'--', 'color':'m'})
plt.show()

### 4-3. Beta-coefficients of LASSO models

In [ ]:
#Female model
#Import the cleaned beta-coefficients of LASSO
fileDir = './ExportData/'
fileName = '210105_Biological-BMI-paper_baseline-LASSO_combiBMI-Female_LASSO-bcoefs.tsv'
combiBMI_F_bcoefs = pd.read_csv(fileDir+fileName, sep='\t').set_index('Variable')
print('Variables:', len(combiBMI_F_bcoefs))

#Variables with non-zero beta-coefficient
tempDF = combiBMI_F_bcoefs.loc[combiBMI_F_bcoefs['nZeros']!=10]
print('Variables with non-zero beta-coefficient:', len(tempDF),
      '(', len(tempDF)/len(combiBMI_F_bcoefs)*100, '%)')

#Variables with non-zero beta-coefficient in all 10 models
tempDF = combiBMI_F_bcoefs.loc[combiBMI_F_bcoefs['nZeros']==0]
tempDF = tempDF.sort_values(by='MEAN', ascending=False)
print('Variables with non-zero beta-coefficient in all 10 models:', len(tempDF),
      '(', len(tempDF)/len(combiBMI_F_bcoefs)*100, '%)')
display(tempDF)

In [ ]:
#Male model
#Import the cleaned beta-coefficients of LASSO
fileDir = './ExportData/'
fileName = '210105_Biological-BMI-paper_baseline-LASSO_combiBMI-Male_LASSO-bcoefs.tsv'
combiBMI_M_bcoefs = pd.read_csv(fileDir+fileName, sep='\t').set_index('Variable')
print('Variables:', len(combiBMI_M_bcoefs))

#Variables with non-zero beta-coefficient
tempDF = combiBMI_M_bcoefs.loc[combiBMI_M_bcoefs['nZeros']!=10]
print('Variables with non-zero beta-coefficient:', len(tempDF),
      '(', len(tempDF)/len(combiBMI_M_bcoefs)*100, '%)')

#Variables with non-zero beta-coefficient in all 10 models
tempDF = combiBMI_M_bcoefs.loc[combiBMI_M_bcoefs['nZeros']==0]
tempDF = tempDF.sort_values(by='MEAN', ascending=False)
print('Variables with non-zero beta-coefficient in all 10 models:', len(tempDF),
      '(', len(tempDF)/len(combiBMI_M_bcoefs)*100, '%)')
display(tempDF)

In [ ]:
#Both sex model
#Import the cleaned beta-coefficients of LASSO
fileDir = './ExportData/'
fileName = '210105_Biological-BMI-paper_baseline-LASSO_combiBMI-BothSex_LASSO-bcoefs.tsv'
combiBMI_B_bcoefs = pd.read_csv(fileDir+fileName, sep='\t').set_index('Variable')
print('Variables:', len(combiBMI_B_bcoefs))

#Variables with non-zero beta-coefficient
tempDF = combiBMI_B_bcoefs.loc[combiBMI_B_bcoefs['nZeros']!=10]
print('Variables with non-zero beta-coefficient:', len(tempDF),
      '(', len(tempDF)/len(combiBMI_B_bcoefs)*100, '%)')

#Variables with non-zero beta-coefficient in all 10 models
tempDF = combiBMI_B_bcoefs.loc[combiBMI_B_bcoefs['nZeros']==0]
tempDF = tempDF.sort_values(by='MEAN', ascending=False)
print('Variables with non-zero beta-coefficient in all 10 models:', len(tempDF),
      '(', len(tempDF)/len(combiBMI_B_bcoefs)*100, '%)')
display(tempDF)

### 4-4. Variables with non-zero beta-coefficient in all 10 models

In [ ]:
#Variables with non-zero beta-coefficient in all 10 models
tempS1 = set(combiBMI_F_bcoefs.loc[combiBMI_F_bcoefs['nZeros']==0].index)
tempS2 = set(combiBMI_M_bcoefs.loc[combiBMI_M_bcoefs['nZeros']==0].index)
tempS3 = set(combiBMI_B_bcoefs.loc[combiBMI_B_bcoefs['nZeros']==0].index)

#Common the robust variables
var_111 = tempS1 & tempS2 & tempS3
print('Common variables with non-zero beta-coefficient in all 10 models:\n', var_111)
var_110 = (tempS1 & tempS2) - var_111
var_101 = (tempS1 & tempS3) - var_111
var_011 = (tempS2 & tempS3) - var_111
var_100 = tempS1 - (var_110 | var_101 | var_111)
var_010 = tempS2 - (var_110 | var_011 | var_111)
var_001 = tempS3 - (var_101 | var_011 | var_111)

#The number of each subset
var_subset = [len(var_100), len(var_010), len(var_110), len(var_001),
              len(var_101), len(var_011), len(var_111)]
print('Each subset:', var_subset)

#Venn diagram
sns.set(font='Arial', context='talk')
venn3(subsets=var_subset, set_labels=('Female', 'Male', 'Both sex'), set_colors=('r', 'b', 'g'), alpha=0.4)
venn3_circles(subsets=var_subset)
plt.title('Robust variables\n—Combined omics—', fontdict={'fontsize':24})
plt.show()

In [ ]:
#Variables with non-zero beta-coefficient in all 10 models
tempS1 = set(combiBMI_F_bcoefs.loc[combiBMI_F_bcoefs['nZeros']==0].index)
tempS2 = set(combiBMI_M_bcoefs.loc[combiBMI_M_bcoefs['nZeros']==0].index)
tempS3 = set(combiBMI_B_bcoefs.loc[combiBMI_B_bcoefs['nZeros']==0].index)

#Union of the robust variables
tempL = list(tempS1 | tempS2 | tempS3)
print('Union of the variables with non-zero beta-coefficient in all 10 models:', len(tempL))

#Merge (use original bcoefs to rescue ones with zeros in any 10 models)
tempDF1 = combiBMI_F_bcoefs.loc[combiBMI_F_bcoefs.index.isin(tempL)].drop(columns=['MEAN', 'SD', 'nZeros'])
tempDF2 = combiBMI_M_bcoefs.loc[combiBMI_M_bcoefs.index.isin(tempL)].drop(columns=['MEAN', 'SD', 'nZeros'])
tempDF3 = combiBMI_B_bcoefs.loc[combiBMI_B_bcoefs.index.isin(tempL)].drop(columns=['MEAN', 'SD', 'nZeros'])
tempDF = pd.concat([tempDF1, tempDF2, tempDF3], axis=0)
tempDF['Model'] = np.repeat(['Female', 'Male', 'Both sex'], [len(tempDF1), len(tempDF2), len(tempDF3)])

#tidyr::gather in R
tempDF1 = tempDF.reset_index().melt(var_name='LASSO', value_name='bcoef', id_vars=['Variable', 'Model'])

#Order for plot and category for shading
tempDF2 = tempDF1.loc[tempDF1['Model']=='Both sex'].groupby('Variable', as_index=True).agg({'bcoef': np.mean})
tempDF2 = tempDF2.sort_values(by='bcoef', ascending=False)
tempDF2 = pd.DataFrame(index=tempDF2.index)
tempL1 = []
tempL2 = []
for row_n in tempDF2.index.tolist():
    if row_n in metDF_B.columns.tolist():
        tempL1.append('Metabolomics')
        tempL2.append('b')
    elif row_n in chemDF_B.columns.tolist():
        tempL1.append('Clinical labs')
        tempL2.append('g')
    elif row_n in protDF_B.columns.tolist():
        tempL1.append('Proteomics')
        tempL2.append('r')
    else:
        tempL1.append('Error?')
        tempL2.append('k')
tempDF2['Category'] = tempL1
tempDF2['Color'] = tempL2

#Plot
sns.set(style='ticks', font='Arial', context='talk')
sns.catplot(data=tempDF1, y='Variable', order=tempDF2.index.tolist(), x='bcoef', kind='box',
                hue='Model', palette='Set1', dodge=True, height=40, aspect=0.35,
                showfliers=True, flierprops={'marker':'o', 'markerfacecolor':'gray', 'alpha':0.4})
plt.xlabel(r'$\beta$'+'-coefficient in LASSO models')
plt.ylabel('')
tempA = np.arange(-0.04, 0.061, 0.02)
for coord in tempA:
    plt.axvline(x=coord, **{'linestyle':'--', 'color':'gray'})
for row_i in range(len(tempDF2)):
    if row_i%2 == 0:
        plt.axhspan(ymin=row_i-0.5, ymax=row_i+0.5, facecolor=tempDF2.iloc[row_i]['Color'], alpha=0.1)
    else:
        plt.axhspan(ymin=row_i-0.5, ymax=row_i+0.5, facecolor=tempDF2.iloc[row_i]['Color'], alpha=0.2)
plt.show()

In [ ]:
#Both sex model

#Variables with non-zero beta-coefficient in all 10 models
tempDF = combiBMI_B_bcoefs.loc[combiBMI_B_bcoefs['nZeros']==0].sort_values(by='MEAN', ascending=False)
tempDF = tempDF.drop(columns=['MEAN', 'SD', 'nZeros'])
print('Variables with non-zero beta-coefficient in all 10 models:', len(tempDF))

#tidyr::gather in R
tempDF1 = tempDF.reset_index().melt(var_name='LASSO', value_name='bcoef', id_vars=['Variable'])

#Category for shading
tempDF2 = combiBMI_B_bcoefs.loc[combiBMI_B_bcoefs['nZeros']==0].sort_values(by='MEAN', ascending=False)
tempDF2 = pd.DataFrame(index=tempDF2.index)
tempL1 = []
tempL2 = []
for row_n in tempDF2.index.tolist():
    if row_n in metDF_B.columns.tolist():
        tempL1.append('Metabolomics')
        tempL2.append('b')
    elif row_n in chemDF_B.columns.tolist():
        tempL1.append('Clinical labs')
        tempL2.append('g')
    elif row_n in protDF_B.columns.tolist():
        tempL1.append('Proteomics')
        tempL2.append('r')
    else:
        tempL1.append('Error?')
        tempL2.append('k')
tempDF2['Category'] = tempL1
tempDF2['Color'] = tempL2

#Plot
sns.set(style='ticks', font='Arial', context='talk')
plt.figure(figsize=(5, 38))
p = sns.boxplot(data=tempDF1, y='Variable', x='bcoef', palette=tempDF2['Color'], dodge=False,
                showfliers=True, flierprops={'marker':'o', 'markerfacecolor':'gray', 'alpha':0.4},
                showcaps=True, notch=False)
p.grid(axis='x', linestyle='--', color='black')
sns.despine()
plt.ylabel('')
plt.xlabel(r'$\beta$'+'-coefficient in LASSO model')
for row_i in range(len(tempDF2)):
    if row_i%2 == 0:
        plt.axhspan(ymin=row_i-0.5, ymax=row_i+0.5, facecolor=tempDF2.iloc[row_i]['Color'], alpha=0.4, zorder=0)
    else:
        plt.axhspan(ymin=row_i-0.5, ymax=row_i+0.5, facecolor=tempDF2.iloc[row_i]['Color'], alpha=0.4, zorder=0)
##Save
fileDir = './ExportFigures/'
fileName = '210823_Biological-BMI-paper_LASSO-bcoef_combiBMI-BothSex-bcoef_non-zero-in-all.tif'
plt.gcf().savefig(fileDir+fileName, dpi=300, bbox_inches='tight', pad_inches=0,
                  pil_kwargs={'compression':'tiff_lzw'})
plt.show()

### 4-5. Correlation b/w pairwise variables with non-zero beta coefficient in all 10 models

In [ ]:
#Both sex model
print('Both sex model:')

#Variables with non-zero beta-coefficient in all 10 models
tempL = combiBMI_B_bcoefs.loc[combiBMI_B_bcoefs['nZeros']==0].index.tolist()

#Compute correlation matrix
tempDF = combiDF_B[tempL]
tempDF = tempDF.corr(method='pearson')
print('• Combinations:', int(len(tempDF)*(len(tempDF)-1)/2))

#Extract lower triangle matrix
tempDF = tempDF.where(np.tril(np.ones(tempDF.shape), k=-1).astype(np.bool), other=np.nan)
tempDF.index.set_names('Variable1', inplace=True)
tempDF = tempDF.reset_index().melt(var_name='Variable2', value_name='Pearson_r', id_vars=['Variable1'])
tempDF = tempDF.dropna()
print('• nrows:', len(tempDF))
print('• |Pearson\'s r| > 0.8:', len(tempDF.loc[abs(tempDF['Pearson_r'])>0.8]))
display(tempDF.loc[abs(tempDF['Pearson_r'])>0.8])

#Distribution
sns.set(style='ticks', font='Arial', context='talk')
plt.figure(figsize=(4, 3))
sns.distplot(tempDF['Pearson_r'], color='m').set(xlim=(-1, 1), xticks=np.arange(-1, 1.1, 0.5))
sns.despine()
plt.ylabel('Density')
plt.xlabel('Pearson\'s '+r'$r$')
plt.axvline(x=tempDF['Pearson_r'].max(), **{'linestyle':'--', 'color':'m'})
plt.axvline(x=tempDF['Pearson_r'].min(), **{'linestyle':'--', 'color':'m'})
plt.show()

print('Variables with non-zero beta-coefficient in all 10 models:', len(tempL))
tempDF = combiDF_B[tempL]
tempDF = tempDF.corr(method='pearson')
#Category
tempS = pd.Series(index=tempDF.index, name='Omics category')
for row_n in tempS.index.tolist():
    if row_n in metDF_B.columns.tolist():
        tempS[row_n] = 'tab:blue'
    elif row_n in chemDF_B.columns.tolist():
        tempS[row_n] = 'tab:green'
    elif row_n in protDF_B.columns.tolist():
        tempS[row_n] = 'tab:red'
    else:#Just in case
        tempS[row_n] = 'k'
#Clustermap
sns.set(style='ticks', font='Arial', context='notebook')
cmap = sns.diverging_palette(220, 20, as_cmap=True)
cm = sns.clustermap(tempDF, method='ward', metric='euclidean', cmap=cmap,
                    row_cluster=True, col_cluster=True, row_linkage=None, col_linkage=None,
                    row_colors=tempS, col_colors=tempS, xticklabels=True, yticklabels=True,
                    dendrogram_ratio=(0.1, 0.1), colors_ratio=(0.01, 0.01),
                    cbar_pos=(0.86, 0.105, 0.12, 0.02), cbar_kws={"orientation": "horizontal"},
                    figsize=(28, 28), **{'center':0, 'vmin':-1, 'vmax':1})
cm.cax.set_title('Pearson\'s '+r'$r$', size='xx-large', verticalalignment='bottom')
cm.cax.tick_params(labelsize='x-large')
bottom, top = cm.ax_heatmap.get_ylim()
cm.ax_heatmap.set_ylim(bottom + 0.5, top - 0.5)##To avoid half cut of first and last rows
hm = cm.ax_heatmap.get_position()
rd = cm.ax_row_dendrogram.get_position()
cd = cm.ax_col_dendrogram.get_position()
cm.ax_heatmap.set_position([hm.x0, hm.y0, hm.width, hm.height])
cm.ax_row_dendrogram.set_position([rd.x0+rd.width*0.5, rd.y0, rd.width*0.5, rd.height])
cm.ax_col_dendrogram.set_position([cd.x0, cd.y0, cd.width, cd.height*0.5])
##row/column color bar legend (axis is same with cm.cax!)
rowcol_legend1 = mpatches.Patch(color='tab:blue', label='Metabolomics')
rowcol_legend2 = mpatches.Patch(color='tab:red', label='Proteomics')
rowcol_legend3 = mpatches.Patch(color='tab:green', label='Clinical labs')
plt.legend(handles=[rowcol_legend1, rowcol_legend2, rowcol_legend3], fontsize='x-large',
           title=tempS.name, title_fontsize='xx-large',
           bbox_to_anchor=(0.5, 0), loc='upper center', borderaxespad=3, frameon=False)
##Save
fileDir = './ExportFigures/'
fileName = '210105_Biological-BMI-paper_LASSO-bcoef_clustermap_combiBMI-BothSex-non-zero-bcoef-in-all.tif'
plt.gcf().savefig(fileDir+fileName, dpi=300, bbox_inches='tight', pad_inches=0,
                  pil_kwargs={'compression':'tiff_lzw'})
plt.show()

### 4-6. Explained variance of variables with non-zero beta-coefficient

> OLS linear regression model for significance:  
> ***BMI ~ b0 + b1\*Variable + b2\*Sex + b3\*BaseAge + b4\*PCs***  
> Univariate model:  
> ***BMI ~ b0 + b1\*Variable***  
> –> Biological BMI is included for comparison.  

In [ ]:
#Female model
print('Female model:')

#Variables with non-zero beta-coefficient at least 1 of 10 models
tempL = combiBMI_F_bcoefs.loc[combiBMI_F_bcoefs['nZeros']!=10].index.tolist()

#Prepare dataframe for OLS regressions
tempL = [item for sublist in [tempL, covarL, ['log_BaseCombiBMI']] for item in sublist]
tempDF = combiDF_F[tempL]
tempDF = tempDF.rename(columns={'log_BaseCombiBMI': 'CombiBMI model'})

#Extract independent variables in OLS regressions (including combiBMI)
tempL = tempDF.drop(columns=covarL).columns.tolist()

#Perform OLS regressions
tempL1 = []#For R2
tempL2 = []#For beta-coefficient
tempL3 = []#For SE of beta-coefficient
tempL4 = []#For p-value
tempL5 = []#For R2 in univariate model
t_start = time.time()
for var in tempL:
    tempDF_ols = tempDF[[item for sublist in [covarL, [var]] for item in sublist]]
    tempDF_ols = tempDF_ols.rename(columns={var: 'Variable'})
    #Add a constant for the intercept -> Similar to R, smf automatically add a constant

    #Univariate model
    #Fit model
    formula = 'log_BaseBMI ~ Variable'
    resOLS = smf.ols(formula, data=tempDF_ols).fit()
    #Save R2 [%]
    tempL5.append(resOLS.rsquared*100)
    
    #Full model
    #Fit model (Sex is invariant in this model and eliminated)
    formula = 'log_BaseBMI ~ Variable + BaseAge + PC1 + PC2 + PC3 + PC4 + PC5'
    resOLS = smf.ols(formula, data=tempDF_ols).fit()
    #Save R2 [%]
    tempL1.append(resOLS.rsquared*100)
    #Save beta-coefficient of the variable
    tempL2.append(resOLS.params['Variable'])
    tempL3.append(resOLS.bse['Variable'])
    #Save p-value of the variable
    tempL4.append(resOLS.pvalues['Variable'])
t_elapsed = time.time() - t_start
print('Elapsed time for ', len(tempL), 'OLS regressions (including combiBMI):',
      round(t_elapsed//60), 'min', round(t_elapsed%60, 1), 'sec')

#Clean the results
tempDF = pd.DataFrame({'UnivarR2':tempL5, 'R2':tempL1, 'OLSbcoef':tempL2, 'OLSbcoefSE':tempL3, 'Pval':tempL4},
                      index=tempL)
tempDF.index.set_names('Variable', inplace=True)
##P-value adjustment by using Benjamini–Hochberg method
tempDF['AdjPval'] = multi.multipletests(tempDF['Pval'], alpha=0.05, method='fdr_bh',
                                        is_sorted=False, returnsorted=False)[1]
##Add the LASSO results
tempDF['LASSObcoef'] = combiBMI_F_bcoefs.loc[combiBMI_F_bcoefs.index.isin(tempL)]['MEAN']#NaN in combiBMI reference
tempDF['LASSOnZeros'] = combiBMI_F_bcoefs.loc[combiBMI_F_bcoefs.index.isin(tempL)]['nZeros']#NaN in combiBMI reference
tempDF = tempDF.sort_values(by='AdjPval', ascending=True)

#Save
fileDir = './ExportData/'
fileName = '210105_Biological-BMI-paper_LASSO-bcoef_OLS_combiBMI-Female.tsv'
tempDF.to_csv(fileDir+fileName, sep='\t', index=True)

#Extact significant variables
tempDF = tempDF[tempDF['AdjPval']<0.05]
print('Variables significantly associated with BMI (FDR < 0.05):', len(tempDF)-1)#combiBMI reference

#Top 30 (+ combiBMI)
topX = 30+1
display(tempDF.iloc[0:topX])
##Category
tempL = []
for row_n in tempDF.index.tolist():
    if row_n=='CombiBMI model':
        tempL.append('Positive association')
    else:
        if tempDF.loc[row_n, 'OLSbcoef'] > 0:
            tempL.append('Positive association')
        elif tempDF.loc[row_n, 'OLSbcoef'] < 0:
            tempL.append('Negative association')
        else:
            tempL.append('Error')
tempDF['Association'] = tempL
##Plot
tempDF = tempDF.reset_index()
sns.set(style='ticks', font='Arial', context='talk')
plt.figure(figsize=(10, 4))
p = sns.barplot(data=tempDF.iloc[0:topX], y='UnivarR2', x='Variable', hue='Association', dodge=False,
                palette={'Positive association':'tab:red', 'Negative association':'tab:blue'}, edgecolor='k')
p.set(ylim=(0, 80), yticks=np.arange(0, 80, 15))
p.grid(axis='y', linestyle='--', color='black')
sns.despine()
plt.xticks(rotation=90, horizontalalignment='center')
plt.xlabel('')
plt.ylabel('Ratio of explained variance [%]')
plt.legend(loc='upper right')
plt.show()

In [ ]:
#Male model
print('Male model:')

#Variables with non-zero beta-coefficient at least 1 of 10 models
tempL = combiBMI_M_bcoefs.loc[combiBMI_M_bcoefs['nZeros']!=10].index.tolist()

#Prepare dataframe for OLS regressions
tempL = [item for sublist in [tempL, covarL, ['log_BaseCombiBMI']] for item in sublist]
tempDF = combiDF_M[tempL]
tempDF = tempDF.rename(columns={'log_BaseCombiBMI': 'CombiBMI model'})

#Extract independent variables in OLS regressions (including combiBMI)
tempL = tempDF.drop(columns=covarL).columns.tolist()

#Perform OLS regressions
tempL1 = []#For R2
tempL2 = []#For beta-coefficient
tempL3 = []#For SE of beta-coefficient
tempL4 = []#For p-value
tempL5 = []#For R2 in univariate model
t_start = time.time()
for var in tempL:
    tempDF_ols = tempDF[[item for sublist in [covarL, [var]] for item in sublist]]
    tempDF_ols = tempDF_ols.rename(columns={var: 'Variable'})
    #Add a constant for the intercept -> Similar to R, smf automatically add a constant

    #Univariate model
    #Fit model
    formula = 'log_BaseBMI ~ Variable'
    resOLS = smf.ols(formula, data=tempDF_ols).fit()
    #Save R2 [%]
    tempL5.append(resOLS.rsquared*100)
    
    #Full model
    #Fit model (Sex is invariant in this model and eliminated)
    formula = 'log_BaseBMI ~ Variable + BaseAge + PC1 + PC2 + PC3 + PC4 + PC5'
    resOLS = smf.ols(formula, data=tempDF_ols).fit()
    #Save R2 [%]
    tempL1.append(resOLS.rsquared*100)
    #Save beta-coefficient of the variable
    tempL2.append(resOLS.params['Variable'])
    tempL3.append(resOLS.bse['Variable'])
    #Save p-value of the variable
    tempL4.append(resOLS.pvalues['Variable'])
t_elapsed = time.time() - t_start
print('Elapsed time for ', len(tempL), 'OLS regressions (including combiBMI):',
      round(t_elapsed//60), 'min', round(t_elapsed%60, 1), 'sec')

#Clean the results
tempDF = pd.DataFrame({'UnivarR2':tempL5, 'R2':tempL1, 'OLSbcoef':tempL2, 'OLSbcoefSE':tempL3, 'Pval':tempL4},
                      index=tempL)
tempDF.index.set_names('Variable', inplace=True)
##P-value adjustment by using Benjamini–Hochberg method
tempDF['AdjPval'] = multi.multipletests(tempDF['Pval'], alpha=0.05, method='fdr_bh',
                                        is_sorted=False, returnsorted=False)[1]
##Add the LASSO results
tempDF['LASSObcoef'] = combiBMI_M_bcoefs.loc[combiBMI_M_bcoefs.index.isin(tempL)]['MEAN']#NaN in combiBMI reference
tempDF['LASSOnZeros'] = combiBMI_M_bcoefs.loc[combiBMI_M_bcoefs.index.isin(tempL)]['nZeros']#NaN in combiBMI reference
tempDF = tempDF.sort_values(by='AdjPval', ascending=True)

#Save
fileDir = './ExportData/'
fileName = '210105_Biological-BMI-paper_LASSO-bcoef_OLS_combiBMI-Male.tsv'
tempDF.to_csv(fileDir+fileName, sep='\t', index=True)

#Extact significant variables
tempDF = tempDF[tempDF['AdjPval']<0.05]
print('Variables significantly associated with BMI (FDR < 0.05):', len(tempDF)-1)#combiBMI reference

#Top 30 (+ combiBMI)
topX = 30+1
display(tempDF.iloc[0:topX])
##Category
tempL = []
for row_n in tempDF.index.tolist():
    if row_n=='CombiBMI model':
        tempL.append('Positive association')
    else:
        if tempDF.loc[row_n, 'OLSbcoef'] > 0:
            tempL.append('Positive association')
        elif tempDF.loc[row_n, 'OLSbcoef'] < 0:
            tempL.append('Negative association')
        else:
            tempL.append('Error')
tempDF['Association'] = tempL
##Plot
tempDF = tempDF.reset_index()
sns.set(style='ticks', font='Arial', context='talk')
plt.figure(figsize=(10, 4))
p = sns.barplot(data=tempDF.iloc[0:topX], y='UnivarR2', x='Variable', hue='Association', dodge=False,
                palette={'Positive association':'tab:red', 'Negative association':'tab:blue'}, edgecolor='k')
p.set(ylim=(0, 80), yticks=np.arange(0, 80, 15))
p.grid(axis='y', linestyle='--', color='black')
sns.despine()
plt.xticks(rotation=90, horizontalalignment='center')
plt.xlabel('')
plt.ylabel('Ratio of explained variance [%]')
plt.legend(loc='upper right')
plt.show()

In [ ]:
#Both sex model
print('Both sex model:')

#Variables with non-zero beta-coefficient at least 1 of 10 models
tempL = combiBMI_B_bcoefs.loc[combiBMI_B_bcoefs['nZeros']!=10].index.tolist()

#Prepare dataframe for OLS regressions
tempL = [item for sublist in [tempL, covarL, ['log_BaseCombiBMI']] for item in sublist]
tempDF = combiDF_B[tempL]
tempDF = tempDF.rename(columns={'log_BaseCombiBMI': 'CombiBMI model'})

#Extract independent variables in OLS regressions (including combiBMI)
tempL = tempDF.drop(columns=covarL).columns.tolist()

#Perform OLS regressions
tempL1 = []#For R2
tempL2 = []#For beta-coefficient
tempL3 = []#For SE of beta-coefficient
tempL4 = []#For p-value
tempL5 = []#For R2 in univariate model
t_start = time.time()
for var in tempL:
    tempDF_ols = tempDF[[item for sublist in [covarL, [var]] for item in sublist]]
    tempDF_ols = tempDF_ols.rename(columns={var: 'Variable'})
    #Add a constant for the intercept -> Similar to R, smf automatically add a constant

    #Univariate model
    #Fit model
    formula = 'log_BaseBMI ~ Variable'
    resOLS = smf.ols(formula, data=tempDF_ols).fit()
    #Save R2 [%]
    tempL5.append(resOLS.rsquared*100)
    
    #Full model
    #Fit model
    formula = 'log_BaseBMI ~ Variable + C(Sex) + BaseAge + PC1 + PC2 + PC3 + PC4 + PC5'
    resOLS = smf.ols(formula, data=tempDF_ols).fit()
    #Save R2 [%]
    tempL1.append(resOLS.rsquared*100)
    #Save beta-coefficient of the variable
    tempL2.append(resOLS.params['Variable'])
    tempL3.append(resOLS.bse['Variable'])
    #Save p-value of the variable
    tempL4.append(resOLS.pvalues['Variable'])
t_elapsed = time.time() - t_start
print('Elapsed time for ', len(tempL), 'OLS regressions (including combiBMI):',
      round(t_elapsed//60), 'min', round(t_elapsed%60, 1), 'sec')

#Clean the results
tempDF = pd.DataFrame({'UnivarR2':tempL5, 'R2':tempL1, 'OLSbcoef':tempL2, 'OLSbcoefSE':tempL3, 'Pval':tempL4},
                      index=tempL)
tempDF.index.set_names('Variable', inplace=True)
##P-value adjustment by using Benjamini–Hochberg method
tempDF['AdjPval'] = multi.multipletests(tempDF['Pval'], alpha=0.05, method='fdr_bh',
                                        is_sorted=False, returnsorted=False)[1]
##Add the LASSO results
tempDF['LASSObcoef'] = combiBMI_B_bcoefs.loc[combiBMI_B_bcoefs.index.isin(tempL)]['MEAN']#NaN in combiBMI reference
tempDF['LASSOnZeros'] = combiBMI_B_bcoefs.loc[combiBMI_B_bcoefs.index.isin(tempL)]['nZeros']#NaN in combiBMI reference
tempDF = tempDF.sort_values(by='AdjPval', ascending=True)

#Save
fileDir = './ExportData/'
fileName = '210105_Biological-BMI-paper_LASSO-bcoef_OLS_combiBMI-BothSex.tsv'
tempDF.to_csv(fileDir+fileName, sep='\t', index=True)

#Extact significant variables
tempDF = tempDF[tempDF['AdjPval']<0.05]
print('Variables significantly associated with BMI (FDR < 0.05):', len(tempDF)-1)#combiBMI reference

#Top 30 (+ combiBMI)
topX = 30+1
display(tempDF.iloc[0:topX])
##Category
tempL = []
for row_n in tempDF.index.tolist():
    if row_n=='CombiBMI model':
        tempL.append('Positive association')
    else:
        if tempDF.loc[row_n, 'OLSbcoef'] > 0:
            tempL.append('Positive association')
        elif tempDF.loc[row_n, 'OLSbcoef'] < 0:
            tempL.append('Negative association')
        else:
            tempL.append('Error')
tempDF['Association'] = tempL
##Plot
tempDF = tempDF.reset_index()
sns.set(style='ticks', font='Arial', context='talk')
plt.figure(figsize=(10, 4))
p = sns.barplot(data=tempDF.iloc[0:topX], y='UnivarR2', x='Variable', hue='Association', dodge=False,
                palette={'Positive association':'tab:red', 'Negative association':'tab:blue'}, edgecolor='k')
p.set(ylim=(0, 80), yticks=np.arange(0, 80, 15))
p.grid(axis='y', linestyle='--', color='black')
sns.despine()
plt.xticks(rotation=90, horizontalalignment='center')
plt.xlabel('')
plt.ylabel('Ratio of explained variance [%]')
plt.legend(loc='upper right')
##Save
fileDir = './ExportFigures/'
fileName = '210105_Biological-BMI-paper_LASSO-bcoef_OLS_combiBMI-BothSex_top30.tif'
plt.gcf().savefig(fileDir+fileName, dpi=300, bbox_inches='tight', pad_inches=0,
                  pil_kwargs={'compression':'tiff_lzw'})
plt.show()

## 5. Comparison b/w omics

### 5-1. Pearson's r

#### 5-1-1. Actual values in LASSO models

In [ ]:
#Both sex cohort
print('All variables')
tempDF = pd.DataFrame(columns=['Category', 'Variable1', 'Variable2', 'Pearson_r'])
tempD = {'Metabolomics':'b', 'Proteomics':'r', 'Clinical labs':'g', 'Combined omics':'m'}
tempL1 = [metDF_B, protDF_B, chemDF_B, combiDF_B]
tempL2 = ['log_BaseMetBMI', 'log_BaseProtBMI', 'log_BaseChemBMI', 'log_BaseCombiBMI']
countL1 = []
countL2 = []
for df_i in range(len(tempL1)):
    print(list(tempD.keys())[df_i])
    #Compute correlation matrix
    tempL = [item for sublist in [covarL, [tempL2[df_i]]] for item in sublist]
    tempDF1 = tempL1[df_i].drop(columns=tempL)
    tempDF1 = tempDF1.corr(method='pearson')
    print(' - Combinations:', int(len(tempDF1)*(len(tempDF1)-1)/2))
    #Extract lower triangle matrix
    tempDF1 = tempDF1.where(np.tril(np.ones(tempDF1.shape), k=-1).astype(np.bool), other=np.nan)
    tempDF1.index.set_names('Variable1', inplace=True)
    tempDF1 = tempDF1.reset_index().melt(var_name='Variable2', value_name='Pearson_r', id_vars=['Variable1'])
    tempDF1 = tempDF1.dropna()
    print(' - nrows:', len(tempDF1))
    countL1.append(len(tempDF1))
    print(' - |Pearson\'s r| > 0.8:', len(tempDF1.loc[abs(tempDF1['Pearson_r'])>0.8]))
    countL2.append(len(tempDF1.loc[abs(tempDF1['Pearson_r'])>0.8]))
    #Clean
    tempDF1['Category'] = list(tempD.keys())[df_i]
    tempDF = pd.concat([tempDF, tempDF1], axis=0)
print('Confirmation')
display(tempDF['Category'].value_counts())

#Distribution with violinplot
sns.set(style='ticks', font='Arial', context='talk')
plt.figure(figsize=(4, 3))
ax = sns.violinplot(data=tempDF, x='Pearson_r', y='Category',
                    order=list(tempD.keys()), palette=tempD, dodge=False, scale='width', inner='box')
ax.set(xlim=(-1, 1), xticks=np.arange(-1, 1.1, 0.5))
sns.despine()
##Add annotation
offset_x = 1.75
ax.annotate('| Pearson\'s '+r'$r$'+' | > 0.8', (offset_x, -1), fontsize='medium', annotation_clip=False,
            verticalalignment='center', horizontalalignment='center')
for df_i in range(len(tempD.keys())):
    ax.annotate(f'{countL2[df_i]:,}'+' / '+f'{countL1[df_i]:,}'+' pairs',
                (offset_x, df_i), fontsize='small', annotation_clip=False,
                verticalalignment='center', horizontalalignment='center')
plt.xlabel('Pearson\'s '+r'$r$')
plt.ylabel('')
##Save
fileDir = './ExportFigures/'
fileName = '210105_Biological-BMI-paper_LASSO-bcoef_Pearsonr-BothSex_all-vars-pair.tif'
plt.gcf().savefig(fileDir+fileName, dpi=300, bbox_inches='tight', pad_inches=0,
                  pil_kwargs={'compression':'tiff_lzw'})
plt.show()

In [ ]:
#Both sex cohort
print('Variables with non-zero beta-coefficient in all 10 models')
tempDF = pd.DataFrame(columns=['Category', 'Variable1', 'Variable2', 'Pearson_r'])
tempD = {'Metabolomics':'b', 'Proteomics':'r', 'Clinical labs':'g', 'Combined omics':'m'}
tempL1 = [metDF_B, protDF_B, chemDF_B, combiDF_B]
tempL2 = [metBMI_B_bcoefs, protBMI_B_bcoefs, chemBMI_B_bcoefs, combiBMI_B_bcoefs]
countL1 = []
countL2 = []
for df_i in range(len(tempL1)):
    print(list(tempD.keys())[df_i])
    ##Variables with non-zero beta-coefficient in all 10 models
    tempDF1 = tempL2[df_i]
    tempL = tempDF1.loc[tempDF1['nZeros']==0].index.tolist()
    #Compute correlation matrix
    tempDF1 = tempL1[df_i].loc[:, tempL]
    tempDF1 = tempDF1.corr(method='pearson')
    print(' - Combinations:', int(len(tempDF1)*(len(tempDF1)-1)/2))
    #Extract lower triangle matrix
    tempDF1 = tempDF1.where(np.tril(np.ones(tempDF1.shape), k=-1).astype(np.bool), other=np.nan)
    tempDF1.index.set_names('Variable1', inplace=True)
    tempDF1 = tempDF1.reset_index().melt(var_name='Variable2', value_name='Pearson_r', id_vars=['Variable1'])
    tempDF1 = tempDF1.dropna()
    print(' - nrows:', len(tempDF1))
    countL1.append(len(tempDF1))
    print(' - |Pearson\'s r| > 0.8:', len(tempDF1.loc[abs(tempDF1['Pearson_r'])>0.8]))
    countL2.append(len(tempDF1.loc[abs(tempDF1['Pearson_r'])>0.8]))
    #Clean
    tempDF1['Category'] = list(tempD.keys())[df_i]
    tempDF = pd.concat([tempDF, tempDF1], axis=0)
print('Confirmation')
display(tempDF['Category'].value_counts())

#Distribution with violinplot
sns.set(style='ticks', font='Arial', context='talk')
plt.figure(figsize=(4, 3))
ax = sns.violinplot(data=tempDF, x='Pearson_r', y='Category',
                    order=list(tempD.keys()), palette=tempD, dodge=False, scale='width', inner='box')
ax.set(xlim=(-1, 1), xticks=np.arange(-1, 1.1, 0.5))
sns.despine()
##Add annotation
offset_x = 1.75
ax.annotate('| Pearson\'s '+r'$r$'+' | > 0.8', (offset_x, -1), fontsize='medium', annotation_clip=False,
            verticalalignment='center', horizontalalignment='center')
for df_i in range(len(tempD.keys())):
    ax.annotate(f'{countL2[df_i]:,}'+' / '+f'{countL1[df_i]:,}'+' pairs',
                (offset_x, df_i), fontsize='small', annotation_clip=False,
                verticalalignment='center', horizontalalignment='center')
plt.xlabel('Pearson\'s '+r'$r$')
plt.ylabel('')
##Save
fileDir = './ExportFigures/'
fileName = '210105_Biological-BMI-paper_LASSO-bcoef_Pearsonr-BothSex_LASSO-vars-pair.tif'
plt.gcf().savefig(fileDir+fileName, dpi=300, bbox_inches='tight', pad_inches=0,
                  pil_kwargs={'compression':'tiff_lzw'})
plt.show()

### 5-2. Variables with non-zero beta-coefficient in all 10 models b/w omics

#### 5-2-1. vs. Metabolomics

In [ ]:
#Variables with non-zero beta-coefficient in all 10 models
tempS1 = set(metBMI_F_bcoefs.loc[metBMI_F_bcoefs['nZeros']==0].index)
tempS2 = set(combiBMI_F_bcoefs.loc[combiBMI_F_bcoefs['nZeros']==0].index)

#Extract analytes in same omics type
tempS2 = tempS2 & set(metDF_F.columns)

#Common variables with non-zero beta-coefficient in all 10 models
tempS3 = tempS1 & tempS2
print('Common variables with non-zero beta-coefficient in all 10 models:\n', tempS3)

#Not common variables with non-zero beta-coefficient in all 10 models
tempS1 = tempS1 - tempS3
tempS2 = tempS2 - tempS3

#The number of each subset
tempL = [len(tempS1), len(tempS2), len(tempS3)]
print('Each subset:', tempL)

#Venn diagram
sns.set(font='Arial', context='talk')
venn2(subsets=tempL, set_labels=('MetBMI', 'CombiBMI'), set_colors=('b', 'm'), alpha=0.7)
venn2_circles(subsets=tempL)
plt.title('—Female model—', fontdict={'fontsize':24})
plt.show()

In [ ]:
#Variables with non-zero beta-coefficient in all 10 models
tempS1 = set(metBMI_M_bcoefs.loc[metBMI_M_bcoefs['nZeros']==0].index)
tempS2 = set(combiBMI_M_bcoefs.loc[combiBMI_M_bcoefs['nZeros']==0].index)

#Extract analytes in same omics type
tempS2 = tempS2 & set(metDF_M.columns)

#Common variables with non-zero beta-coefficient in all 10 models
tempS3 = tempS1 & tempS2
print('Common variables with non-zero beta-coefficient in all 10 models:\n', tempS3)

#Not common variables with non-zero beta-coefficient in all 10 models
tempS1 = tempS1 - tempS3
tempS2 = tempS2 - tempS3

#The number of each subset
tempL = [len(tempS1), len(tempS2), len(tempS3)]
print('Each subset:', tempL)

#Venn diagram
sns.set(font='Arial', context='talk')
venn2(subsets=tempL, set_labels=('MetBMI', 'CombiBMI'), set_colors=('b', 'm'), alpha=0.7)
venn2_circles(subsets=tempL)
plt.title('—Male model—', fontdict={'fontsize':24})
plt.show()

In [ ]:
#Variables with non-zero beta-coefficient in all 10 models
tempS1 = set(metBMI_B_bcoefs.loc[metBMI_B_bcoefs['nZeros']==0].index)
tempS2 = set(combiBMI_B_bcoefs.loc[combiBMI_B_bcoefs['nZeros']==0].index)

#Extract analytes in same omics type
tempS2 = tempS2 & set(metDF_B.columns)

#Common variables with non-zero beta-coefficient in all 10 models
tempS3 = tempS1 & tempS2
print('Common variables with non-zero beta-coefficient in all 10 models:\n', tempS3)

#Not common variables with non-zero beta-coefficient in all 10 models
tempS1 = tempS1 - tempS3
tempS2 = tempS2 - tempS3

#The number of each subset
tempL = [len(tempS1), len(tempS2), len(tempS3)]
print('Each subset:', tempL)

#Venn diagram
sns.set(font='Arial', context='talk')
venn2(subsets=tempL, set_labels=('MetBMI', 'CombiBMI'), set_colors=('b', 'm'), alpha=0.7)
venn2_circles(subsets=tempL)
plt.title('—Both sex model—', fontdict={'fontsize':24})
plt.show()

#### 5-2-2. vs. Clinical labs

In [ ]:
#Variables with non-zero beta-coefficient in all 10 models
tempS1 = set(chemBMI_F_bcoefs.loc[chemBMI_F_bcoefs['nZeros']==0].index)
tempS2 = set(combiBMI_F_bcoefs.loc[combiBMI_F_bcoefs['nZeros']==0].index)

#Extract analytes in same omics type
tempS2 = tempS2 & set(chemDF_F.columns)

#Common variables with non-zero beta-coefficient in all 10 models
tempS3 = tempS1 & tempS2
print('Common variables with non-zero beta-coefficient in all 10 models:\n', tempS3)

#Not common variables with non-zero beta-coefficient in all 10 models
tempS1 = tempS1 - tempS3
tempS2 = tempS2 - tempS3

#The number of each subset
tempL = [len(tempS1), len(tempS2), len(tempS3)]
print('Each subset:', tempL)

#Venn diagram
sns.set(font='Arial', context='talk')
venn2(subsets=tempL, set_labels=('ChemBMI', 'CombiBMI'), set_colors=('g', 'm'), alpha=0.7)
venn2_circles(subsets=tempL)
plt.title('—Female model—', fontdict={'fontsize':24})
plt.show()

In [ ]:
#Variables with non-zero beta-coefficient in all 10 models
tempS1 = set(chemBMI_M_bcoefs.loc[chemBMI_M_bcoefs['nZeros']==0].index)
tempS2 = set(combiBMI_M_bcoefs.loc[combiBMI_M_bcoefs['nZeros']==0].index)

#Extract analytes in same omics type
tempS2 = tempS2 & set(chemDF_M.columns)

#Common variables with non-zero beta-coefficient in all 10 models
tempS3 = tempS1 & tempS2
print('Common variables with non-zero beta-coefficient in all 10 models:\n', tempS3)

#Not common variables with non-zero beta-coefficient in all 10 models
tempS1 = tempS1 - tempS3
tempS2 = tempS2 - tempS3

#The number of each subset
tempL = [len(tempS1), len(tempS2), len(tempS3)]
print('Each subset:', tempL)

#Venn diagram
sns.set(font='Arial', context='talk')
venn2(subsets=tempL, set_labels=('ChemBMI', 'CombiBMI'), set_colors=('g', 'm'), alpha=0.7)
venn2_circles(subsets=tempL)
plt.title('—Male model—', fontdict={'fontsize':24})
plt.show()

In [ ]:
#Variables with non-zero beta-coefficient in all 10 models
tempS1 = set(chemBMI_B_bcoefs.loc[chemBMI_B_bcoefs['nZeros']==0].index)
tempS2 = set(combiBMI_B_bcoefs.loc[combiBMI_B_bcoefs['nZeros']==0].index)

#Extract analytes in same omics type
tempS2 = tempS2 & set(chemDF_B.columns)

#Common variables with non-zero beta-coefficient in all 10 models
tempS3 = tempS1 & tempS2
print('Common variables with non-zero beta-coefficient in all 10 models:\n', tempS3)

#Not common variables with non-zero beta-coefficient in all 10 models
tempS1 = tempS1 - tempS3
tempS2 = tempS2 - tempS3

#The number of each subset
tempL = [len(tempS1), len(tempS2), len(tempS3)]
print('Each subset:', tempL)

#Venn diagram
sns.set(font='Arial', context='talk')
venn2(subsets=tempL, set_labels=('ChemBMI', 'CombiBMI'), set_colors=('g', 'm'), alpha=0.7)
venn2_circles(subsets=tempL)
plt.title('—Both sex model—', fontdict={'fontsize':24})
plt.show()

#### 5-2-3. vs. Proteomics

In [ ]:
#Variables with non-zero beta-coefficient in all 10 models
tempS1 = set(protBMI_F_bcoefs.loc[protBMI_F_bcoefs['nZeros']==0].index)
tempS2 = set(combiBMI_F_bcoefs.loc[combiBMI_F_bcoefs['nZeros']==0].index)

#Extract analytes in same omics type
tempS2 = tempS2 & set(protDF_F.columns)

#Common variables with non-zero beta-coefficient in all 10 models
tempS3 = tempS1 & tempS2
print('Common variables with non-zero beta-coefficient in all 10 models:\n', tempS3)

#Not common variables with non-zero beta-coefficient in all 10 models
tempS1 = tempS1 - tempS3
tempS2 = tempS2 - tempS3

#The number of each subset
tempL = [len(tempS1), len(tempS2), len(tempS3)]
print('Each subset:', tempL)

#Venn diagram
sns.set(font='Arial', context='talk')
venn2(subsets=tempL, set_labels=('ProtBMI', 'CombiBMI'), set_colors=('r', 'm'), alpha=0.7)
venn2_circles(subsets=tempL)
plt.title('—Female model—', fontdict={'fontsize':24})
plt.show()

In [ ]:
#Variables with non-zero beta-coefficient in all 10 models
tempS1 = set(protBMI_M_bcoefs.loc[protBMI_M_bcoefs['nZeros']==0].index)
tempS2 = set(combiBMI_M_bcoefs.loc[combiBMI_M_bcoefs['nZeros']==0].index)

#Extract analytes in same omics type
tempS2 = tempS2 & set(protDF_M.columns)

#Common variables with non-zero beta-coefficient in all 10 models
tempS3 = tempS1 & tempS2
print('Common variables with non-zero beta-coefficient in all 10 models:\n', tempS3)

#Not common variables with non-zero beta-coefficient in all 10 models
tempS1 = tempS1 - tempS3
tempS2 = tempS2 - tempS3

#The number of each subset
tempL = [len(tempS1), len(tempS2), len(tempS3)]
print('Each subset:', tempL)

#Venn diagram
sns.set(font='Arial', context='talk')
venn2(subsets=tempL, set_labels=('ProtBMI', 'CombiBMI'), set_colors=('r', 'm'), alpha=0.7)
venn2_circles(subsets=tempL)
plt.title('—Male model—', fontdict={'fontsize':24})
plt.show()

In [ ]:
#Variables with non-zero beta-coefficient in all 10 models
tempS1 = set(protBMI_B_bcoefs.loc[protBMI_B_bcoefs['nZeros']==0].index)
tempS2 = set(combiBMI_B_bcoefs.loc[combiBMI_B_bcoefs['nZeros']==0].index)

#Extract analytes in same omics type
tempS2 = tempS2 & set(protDF_B.columns)

#Common variables with non-zero beta-coefficient in all 10 models
tempS3 = tempS1 & tempS2
print('Common variables with non-zero beta-coefficient in all 10 models:\n', tempS3)

#Not common variables with non-zero beta-coefficient in all 10 models
tempS1 = tempS1 - tempS3
tempS2 = tempS2 - tempS3

#The number of each subset
tempL = [len(tempS1), len(tempS2), len(tempS3)]
print('Each subset:', tempL)

#Venn diagram
sns.set(font='Arial', context='talk')
venn2(subsets=tempL, set_labels=('ProtBMI', 'CombiBMI'), set_colors=('r', 'm'), alpha=0.7)
venn2_circles(subsets=tempL)
plt.title('—Both sex model—', fontdict={'fontsize':24})
plt.show()